<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-12-production-deploy/lesson-12.2-rag-backend/notebooks/GCP_Capstone_12.2_RAGBackend.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.2 RAG API Backend — The Door, the File, the Candidate
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

The one door every surface on the lane asks through. This notebook is where `services/rag-api` comes from - the heredocs at the end are the kit's source - and the service they build has been live since 6 September and grown by Modules 7 to 11: two identity legs, a retrieval backend, routing with a breaker, an explicit cache, a model that is a setting, a backend that is a setting, a tenant pin, media routes, a guard behind a switch, and a `/version` that names what is serving. The story now runs against it: one question three ways (answered, refused by the roster, refused by the platform), the file in excerpts with the revision's switches read back, `/version` and the candidate revision the gate judges, the stream as the UI reads it, and the smoke run from the clone.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
CANDIDATE_URL = f"https://candidate---documind-api-{NUMBER}.{REGION}.run.app"   # a no-traffic revision: make candidate (an env flip) or make release-candidate (a new image)
QUESTION = "After how many years of continuous service does gratuity become payable?"   # golden lk-16: the Payment of Gratuity Act, in every tenant's corpus

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They land in
# Cloud Logging first (the sink copies them into BigQuery for the view); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: The door
The member answered, the outsider refused by the roster, nobody refused by the platform - and the row.


In [ ]:
# THE DOOR. One question through /v1/query, three ways. As the roster member: 200, an answer with citations, and the row
# it leaves. As the outsider - the account IAM admits and every roster refuses (4.8, 7.2): 403, a real refusal from the
# app. With no token at all: refused before the app sees it, by Cloud Run IAM. Two refusals, two different layers; a
# smoke test that cannot tell them apart is how an incident takes an hour (12.8).
body = {"query": QUESTION, "tenant_id": TENANT, "user_id": "u_12_2", "top_k": 5, "stream": False}
st, ans = api("/v1/query", body)
assert st == 200, (st, ans)
print(f"member  : {st}  answerable={ans['answerable']} citations={len(ans['citations'])} model={ans.get('model')} backend={ans.get('backend')} {ans['latency_ms']} ms")
print("          ", ans["answer"][:150].replace(chr(10), " "), "...")
print("           cites", [c["chunk_id"] for c in ans["citations"]][:3])
assert ans["answerable"] and ans["citations"], "the golden question must be answered with a citation"
st_o, out = api("/v1/query", body, token=id_token_as(OUTSIDER_SA, API_URL))
st_n, none = api("/v1/query", body, token=None)
print(f"outsider: {st_o}  {str(out)[:90]}")
print(f"no token: {st_n}  {'the platform (HTML)' if isinstance(none, str) and '<' in none else none}")
assert st_o == 403 and st_n in (401, 403), (st_o, st_n)
time.sleep(20)
rows = usage_rows(minutes=2, limit=3)
print("\nthe row:", {k: rows[0].get(k) for k in ("tenant", "model", "model_backend", "guard", "cost_usd", "latency_ms", "brain")} if rows else "(not in Logging yet - re-run this line in a minute)")


## Cell 3: The file, in excerpts
The lines that carry the story, from the clone; the switches, from the revision.


In [ ]:
# THE FILE, IN EXCERPTS. The whole of services/rag-api is at the end of this notebook, because that is where the kit
# reads it from. The story is these excerpts from the clone, around the lines that carry it: the query route
# (retrieve -> rerank -> generate, the row), the generator's entry (the model and the backend as settings), the one
# verifier's two legs (shared/iap.py) - and the switches, read off the LIVE revision rather than the file.
print(excerpt("services/rag-api/main.py", "def query(", 1, 24)); print()
print(excerpt("services/rag-api/generator.py", "def generate(", 0, 12)); print()
print(excerpt("shared/iap.py", "def identity(", 0, 16)); print()
print(excerpt("services/rag-api/retriever.py", "def newest_per_source(", 0, 10)); print()
env = service("documind-api")["env"]
switches = {k: env.get(k) for k in ("RETRIEVAL_BACKEND", "RETRIEVAL_CURRENT_ONLY", "RETRIEVAL_GRAPH", "EMBEDDING_MODEL", "EMBEDDING_VERSION", "GENERATOR_MODEL",
                                    "RAG_MODEL_BASE", "ROUTING", "MODEL_BACKEND", "ARMOR", "BUDGET_USD", "GIT_SHA")}
print("the revision's switches:", switches)
# THE VERSIONS VIEW (12 September 2026, deploy/INDEXING.md): the tenant's ledger through the API - which version of each
# document is current, what the last reindex cost, the corpus fingerprint the cache is keyed to. Only for a tenant the
# caller is on the roster of; the UI's Documents page renders the same rows, make sources prints them.
st_s, led = api_get(f"/v1/sources?tenant_id={TENANT}")
if st_s == 200:
    print(f"the ledger: {led.get('versions')} current versions, fingerprint {led.get('fingerprint')}, last event {led.get('last_event')}")
    for s_ in led.get("sources", [])[:6]:
        print(f"  {(s_.get('name') or '')[-40:]:40} {s_.get('status') or '':9} chunks={s_.get('chunks')} reused={s_.get('reused')} "
              f"embedded={s_.get('embedded')} retired={s_.get('retired')} {s_.get('effective_from') or ''}")
else:
    print(f"/v1/sources: HTTP {st_s} - the API on the lane predates the versions view (make build deploy-services SERVICES=api)")
assert env.get("RETRIEVAL_BACKEND", "vector") == "vector", "the deployment's default is Vector Search (vector.tf; make deploy-services) - a tenant's pin, not the deployment, sends a request to the Firestore rung or a managed store"
assert env.get("MODEL_BACKEND", "vertex") == "vertex" and env.get("ARMOR", "off") == "off", "the sessions' lane stays on vertex with the guard off; candidates carry the flips"


## Cell 4: /version and the candidate


In [ ]:
# /VERSION AND THE CANDIDATE. The triple every log line carries - backend, model, prompt, retrieval mode - plus the git
# sha of the image, on the live revision; then the traffic list. A candidate is a revision of this same service that
# takes no traffic and answers on its own URL: make candidate flips an environment variable on the current image (10.1
# judged the tuned model there, 11.4 the self-hosted one), make release-candidate puts a NEW image there (12.7's
# release). Either way the gate runs against it before anyone flips traffic - and the flip is the rollback too.
st, ver = api_get("/version")
print(st, ver)
assert st == 200 and ver.get("git_sha"), ver
print("embedding :", ver.get("embedding", "(a revision before 12 September: not reported)"), "- the pair the worker stamps on every row")
for t in service("documind-api")["traffic"]:
    print(f"  {t.get('revisionName') or 'LATEST':36} {t.get('percent', 0):>3}%  tag={t.get('tag', '-')}  latest={t.get('latestRevision', False)}")
st_c, ver_c = api_get("/version", base=CANDIDATE_URL)
if st_c == 200:
    print("candidate:", ver_c, "- differs from live in:", [k for k in ver if ver.get(k) != ver_c.get(k)] or "nothing")
else:
    print(f"no candidate revision (HTTP {st_c}): make release-candidate GIT_SHA=<sha> or make candidate <flip> tags one")


## Cell 5: The stream, as the UI reads it


In [ ]:
# THE STREAM, AS THE UI READS IT. /v1/stream is Server-Sent Events: the citations first (retrieval knows them before a
# token exists, so the UI can draw the sources while the answer is being written), then tokens, then done with the
# usage and the model that answered. chat.py (12.4) parses exactly these three event kinds; this cell does the same
# with requests, so the contract the UI depends on is asserted here rather than assumed.
r = requests.post(f"{API_URL}/v1/stream", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, stream=True, timeout=120)
assert r.status_code == 200, (r.status_code, r.text[:200])
events, tokens, done, ev = [], [], None, None
for line in r.iter_lines(decode_unicode=True):
    if line.startswith("event: "):
        ev = line[7:]
    elif line.startswith("data: ") and ev:
        payload = json.loads(line[6:])
        events.append(ev)
        if ev == "token":
            tokens.append(payload["t"])
        elif ev == "done":
            done = payload
        elif ev == "citation" and events.count("citation") <= 2:
            print(f"  citation {payload['n']}: {payload['source'].split('/')[-1]} p.{payload['page']} kind={payload.get('kind')}")
summary = {k: done.get(k) for k in ("tokens_in", "tokens_out", "model", "backend", "latency_ms")}
print(f"  {events.count('citation')} citations, {len(tokens)} token frames, done={summary}")
print("  answer:", "".join(tokens)[:140].replace(chr(10), " "), "...")
assert events[0] == "citation" and "token" in events and events[-1] == "done", events[:3] + events[-2:]


## Cell 6: The smoke, run


In [ ]:
# THE SMOKE, RUN. deploy/smoke/smoke.py is what the lane asks itself before a session: health, ready, one real question
# answered with citations (golden lk-16, never a refusal counted as a pass), and - since 12.8 - the same question with
# no token, refused. Run from the clone with the exports the runbook sets; the exit code is the verdict, as in CI.
env = {**os.environ, "DOCUMIND_PROJECT": PROJECT_ID, "DOCUMIND_API_URL": API_URL, "DOCUMIND_TENANT": TENANT, "DOCUMIND_IMPERSONATE_SA": MEMBER_SA}
r = subprocess.run([sys.executable, f"{KIT}/deploy/smoke/smoke.py"], capture_output=True, text=True, env=env, cwd=f"{KIT}/deploy")
print(chr(10).join(l for l in r.stdout.splitlines() if "[" in l or "pass" in l))
assert r.returncode == 0, "the smoke is red: read the FAIL line before anything else"


## Where this goes
- **12.3** reads the rows this door leaves; **12.4** is the client that streams from it; **12.6** turns its guard on, on a candidate.
- **12.7** ships a new image through the candidate this lesson showed - the gate on it, a person, a traffic flip.
- **4.6's graph is a switch on this door (13 September 2026)**: `RETRIEVAL_GRAPH=on|auto` walks the tenant's knowledge graph (`shared/documind_graph.py`, the lesson's `FirestoreGraph` verbatim; `make graph TENANT=` builds it from 12.5's chunks) and puts the chunks its nodes point at in front of the dense pool; `/version` and the usage row (`retrieval_graph`, `graph_chunks`) say so. Off on the lane; `make candidate RETRIEVAL_GRAPH=auto` is where it is judged.
- **4.3's corpus is a backend of this door (P9.4, 13 September 2026)**: `RETRIEVAL_BACKEND=rag_engine` queries the tenant's RAG Engine corpus - the mirror 12.5 keeps of the ledger's current versions - and maps each context to the kit's chunk contract through the version's own row, so the reranker, the packing, the generator and the citations are unchanged; figures and segments still come from the kit's index. A tenant with no corpus, or a store that will not answer, falls back to Firestore with the filters (`rag_engine_fallback`). Since the evening of 13 September 2026 the store is a per-tenant choice: `tenant_settings/{tenant}.retrieval_backend` pins a tenant's store and its `data_region` (`in` | `any`, absent = `in`; `make tenant-policy`) decides whether a managed store may serve it at all - a managed backend for an `in` tenant is the kit's own index with `policy_fallback=1` on the row, never a 500. `/version` reports the default; the usage row carries the `retrieval_backend` that served, `managed_chunks` and `policy_fallback`; `GET /v1/sources` says where each version is held (`mirrored`). **4.4's data store is the fourth backend (R4)**: `RETRIEVAL_BACKEND=vertex_search` searches the tenant's data store by text through its default serving config, as 4.4's cell does, and makes each extractive segment (or the snippet, when the store serves none) a chunk of the same contract through its version's row - the caller's `doc_type` as a filter expression on the store's structData, media from the kit's index, the Firestore rung on no store or an error (`vertex_search_fallback`). `make up` pins zeta to it and acme to `rag_engine` (`make managed-stores`). `make ablate ABLATE_ARGS="--arms all"` measures it, `make candidate RETRIEVAL_BACKEND=rag_engine` judges it.
- **The ANN tier says which rung answered (16 September 2026)**: every chunk `retrieve()` returns carries `found_by` - `vector` for an id `find_neighbors` gave, `firestore` for a row Firestore's own index gave, chosen or fallen into - and the answer carries `stages.retrieval_backend` (the backend chosen for that request) and `stages.vector_chunks` (the index's share of the pool), beside `managed_chunks` and `graph_chunks`. A deployment on `vector` whose answers say `vector_chunks: 0` is answering from the rung beneath, and `make smoke` fails on it. `vector.tf` creates the index EMPTY - no `contents_delta_uri`, the batch input the kit never writes and CreateIndex refuses empty - so `make up` ends with `make vector-status` (0 datapoints until `make ingest-corpus`), `make wait-vectors WANT=200` blocks until the corpus is in, and `make backfill-vectors APPLY=1` streams every current row's stored embedding up when the worker could not.

## ✅ Lesson 12.2 complete
- ✅ One question three ways: 200 with citations, 403 from the roster, refused by the platform with no token
- ✅ The route, the generator and the verifier in excerpts from the clone; the revision's switches read back
- ✅ /version on the live revision; the traffic list; the candidate's URL and what it differs in
- ✅ The SSE contract asserted: citations, tokens, done
- ✅ The smoke run from the clone, exit code and all


## The files this lesson owns
Below are the twelve heredocs the extractor turns into `deploy/services/rag-api/*` and `deploy/commands/lesson-12.2.sh`: the requirements, the settings, the shared answer contract, the request and response schemas, the retriever, the generator, the identity check, the app, the media routes, the Dockerfile, the deploy and the smoke. They are the kit's source and they are unchanged by the rebuild; every Module 10 and 11 seam (routing, the cache, the gateway backend, the tenant pin, the guard) reached them through `tools/readopt.py`. The story above reads the same files from the clone.


In [ ]:
REQUIREMENTS = '''
fastapi==0.141.1
uvicorn[standard]==0.52.4
gunicorn==26.2.0
pydantic==2.13.5
pydantic-settings==2.15.0
google-cloud-aiplatform==1.153.1
google-cloud-firestore==2.30.0
google-genai==2.22.0
google-cloud-discoveryengine==0.13.11
google-cloud-bigquery==3.45.0
# 9.4's Media Studio router (media.py, gap G8): generated assets and signed upload URLs.
google-cloud-storage==3.13.1
# 12.6: the guard, and gen_ai spans. util-genai is what actually reads
# OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT, so pin it too -
# the content-capture default lives there, not in the instrumentation.
google-cloud-modelarmor==0.7.1
opentelemetry-instrumentation-google-genai==1.1b1
opentelemetry-util-genai==1.1b0
google-cloud-logging==3.16.3
opentelemetry-api==1.44.0
opentelemetry-sdk==1.44.0
opentelemetry-exporter-gcp-trace==1.15.0
opentelemetry-instrumentation-fastapi==0.65b0
tenacity==9.1.4
httpx==0.28.1
tiktoken==0.14.0
# 4.6 on the lane, Spanner Graph (16 September 2026): shared/documind_graph.SpannerGraph, GRAPH_BACKEND=spanner
google-cloud-spanner==3.60.0
'''
with open('requirements.txt', 'w') as f: f.write(REQUIREMENTS)
print('requirements.txt written')


In [ ]:
CONFIG_PY = '''
from pydantic import Field, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict

RETRIEVAL_BACKENDS = ("vector", "firestore", "rag_engine", "vertex_search")   # rag_engine: 4.3's corpus (P9.4); vertex_search: 4.4's data store (R4) - each as the retrieval stage
MANAGED_BACKENDS = ("rag_engine", "vertex_search")   # embed and search on their own terms, outside India: a tenant's data_region decides per request
RETRIEVAL_MODES = ("dense", "hybrid")
GRAPH_MODES = ("off", "on", "auto")          # 4.6's graph on the lane (13 September 2026, shared/documind_graph.py)
GRAPH_BACKENDS = ("firestore", "spanner")    # where the graph lives (16 September 2026): Spanner seeds the walk by meaning


def check_retrieval_modes(backend: str, mode: str, graph: str = "off", graph_backend: str = "firestore") -> None:
    """RETRIEVAL_MODE against RETRIEVAL_BACKEND, at startup (12 September 2026, R06). Hybrid is Vector Search's
    HybridQuery (4.5's hybrid.py); Firestore's vector index takes one dense vector and nothing else, so with the
    Firestore backend RETRIEVAL_MODE=hybrid ran dense while every usage row and /version said hybrid. A service that cannot do
    what its environment says must not start: the message names the fix, so it is read on the failed deploy and not
    found in the rows a week later. An unknown value is refused for the same reason - a typo ran dense too.

    What is NOT judged here since 13 September 2026 (evening): whether a managed backend may serve a tenant. That is
    the tenant's data_region (tenant_settings/{tenant}, shared/tenancy.py), asked per request in main.py's
    choose_for - a deployment variable cannot know that one tenant may leave India and another may not."""
    if backend not in RETRIEVAL_BACKENDS or mode not in RETRIEVAL_MODES:
        raise ValueError(f"RETRIEVAL_BACKEND={backend!r} RETRIEVAL_MODE={mode!r}: the backend is one of "
                         f"{'|'.join(RETRIEVAL_BACKENDS)} and the mode one of {'|'.join(RETRIEVAL_MODES)}")
    if graph not in GRAPH_MODES:
        raise ValueError(f"RETRIEVAL_GRAPH={graph!r}: one of {'|'.join(GRAPH_MODES)} - off touches nothing, on walks the tenant's "
                         "graph for every question, auto only for a relational question with a seed entity (4.6's choose_mode)")
    if graph_backend not in GRAPH_BACKENDS:
        raise ValueError(f"GRAPH_BACKEND={graph_backend!r}: one of {'|'.join(GRAPH_BACKENDS)} - firestore is 4.6's store beside the "
                         "chunks; spanner is spanner.tf's Spanner Graph, which seeds by meaning (SPANNER_INSTANCE / SPANNER_DATABASE)")
    if backend in MANAGED_BACKENDS and mode == "hybrid":
        raise ValueError(f"RETRIEVAL_MODE=hybrid needs RETRIEVAL_BACKEND=vector: {backend} embeds and searches on its own "
                         "terms (a managed store has no sparse leg to fuse). Set RETRIEVAL_MODE=dense.")
    if backend == "firestore" and mode == "hybrid":
        raise ValueError("RETRIEVAL_MODE=hybrid needs RETRIEVAL_BACKEND=vector: the Firestore backend is dense-only. "
                         "Set RETRIEVAL_MODE=dense, or RETRIEVAL_BACKEND=vector with the Vector Search endpoint "
                         "vector.tf declares and keep hybrid.")


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")

    project_id: str = Field(alias="GOOGLE_CLOUD_PROJECT")
    region: str = "us-central1"
    india_region: str = "asia-south1"
    # Which store answers the vector query. `vector` is Vector Search with the Firestore
    # fallback beneath it (the chaos rung). `firestore` - the rung on its own - is
    # Firestore's own vector index alone: no endpoint to keep warm, the same tenant
    # pre-filter, the ANN tier left out. Nothing else in the service changes.
    retrieval_backend: str = Field("vector", alias="RETRIEVAL_BACKEND")   # vector | firestore | rag_engine | vertex_search (P9.4, R4): the DEFAULT
    # P9.4 (13 September 2026): the corpora's region (serverless RAG Engine: us-central1 only) and 4.3's cosine-distance
    # threshold on a context. RETRIEVAL_BACKEND is the deployment's default since the evening of that day: the same
    # tenant_settings/{tenant} document that pins a tenant's model (11.4) may pin its retrieval_backend, and its
    # data_region decides whether a managed store may serve it at all - a managed backend for an `in` tenant is the
    # kit's own index with policy_fallback=1 on the row (main.py's choose_for and retrieval_backend_for).
    rag_location: str = Field("us-central1", alias="RAG_LOCATION")
    rag_distance_threshold: float = Field(0.5, alias="RAG_DISTANCE_THRESHOLD")
    # R4 (13 September 2026, evening): 4.4's data stores are global (managed.tf), searched through their default serving
    # config as the lesson's notebook does; the extractive segments asked per result (a plain data store answers with
    # snippets, an Enterprise engine with segments - the backend takes whichever came).
    search_location: str = Field("global", alias="SEARCH_LOCATION")
    search_segments: int = Field(3, alias="SEARCH_SEGMENTS")
    # The ledger (12.5, 11 September 2026): `on` retrieves only chunks the ledger marks current - one
    # version per document. Off until the second vector index is built and the chunks written before
    # the ledger carry the field (make backfill-current); a switch, judged on a candidate like the others.
    retrieval_current_only: str = Field("off", alias="RETRIEVAL_CURRENT_ONLY")   # off | on
    vector_index_endpoint: str = Field("", alias="VECTOR_INDEX_ENDPOINT")
    vector_deployed_index: str = Field("", alias="VECTOR_DEPLOYED_INDEX_ID")
    # THE Firestore collection. Gap G2: 2.3 wrote `knowledge_base`, 4.2/4.5/4.6 read
    # `rag_chunks`, and this service read `chunks` - three names for one corpus, so the
    # teaching lane and the production lane never saw the same documents. Every notebook
    # from 2.3 onward now names this one. services/ingest/indexer.py writes to it.
    chunks_collection: str = "chunks"
    # ONE declared embedding (12 September 2026): EMBEDDING_MODEL and EMBEDDING_VERSION are variables.tf's
    # embedding_model / embedding_version, set on this service AND on the ingest worker by make deploy-services,
    # so the query vector and the document vectors come from one model by construction. The worker stamps the
    # pair on every chunk row; a bump is a planned migration (make reembed, deploy/INDEXING.md), never a silent
    # mismatch. /version reports it beside the model and the prompt.
    embed_model: str = Field("text-embedding-005", alias="EMBEDDING_MODEL")
    embedding_version: str = Field("1", alias="EMBEDDING_VERSION")
    # GENERATOR_MODEL in the environment. A model NAME is served on the global endpoint; a tuned model is
    # an ENDPOINT path (projects/.../locations/us-central1/endpoints/...) and is served on a regional
    # client - generator.py picks by the value (10.1: tuning is regional). RAG_MODEL_BASE names the base
    # a tuned endpoint was tuned from, for pricing (cost.py) and for the cache's model check.
    generator_model: str = "gemini-3.6-flash"
    rag_model_base: str = Field("", alias="RAG_MODEL_BASE")
    # Where a tuned endpoint is served from. Empty: read from the endpoint path (its /locations/<x>/ segment -
    # the first live tuning job put its endpoint in the `us` multi-region, not us-central1, F41). Set it to
    # force a location (global, us-central1) without a code change - a setting, like the model.
    generator_location: str = Field("", alias="GENERATOR_LOCATION")
    # 10.3, behind a flag: ROUTING=on classifies each question (router.py) and lets the budget breaker
    # (breakers.py) pick the tier; off, the generator model above serves everything. The spend the
    # breaker reads is the month's counter in Firestore (budget.py) over BUDGET_USD; SPEND_PCT
    # overrides it for a replay ("what does 85% look like").
    routing: str = Field("off", alias="ROUTING")
    budget_usd: float = Field(100.0, alias="BUDGET_USD")
    spend_pct_override: str = Field("", alias="SPEND_PCT")
    rerank_model: str = "semantic-ranker-fast-004"
    # The Ranking API's deadline (12 September 2026): past it, or on any error, rerank() returns the pool by retrieval
    # score and the row says rerank_fallback=1. Generous beside a 20-record call, which answers well inside a second;
    # a ranker that takes longer is not ranking, it is down, and a worse order beats a hung request and then a 500.
    rerank_timeout_s: float = Field(5.0, alias="RERANK_TIMEOUT_S")
    # The pool the reranker sees - the funnel's width. 20 shipped; evals/ablate.py's "dense 50 -> rerank 5" arm is
    # the measurement that moves it, and the row's rerank_ms / pool columns are what the move costs. An env var, so
    # the move is a number in the service's environment, not a code change. top_k (the request, <= 20) is what comes OUT.
    top_k_retrieve: int = Field(20, alias="TOP_K_RETRIEVE")
    top_k_rerank: int = 5
    max_context_tokens: int = 8000
    # 2048, not 1024: a statute answer with its quotes - and the thinking drawn from the same
    # budget on the 3.x family - outran 1024 on fourteen rows of the second live eval, and each
    # cut-off JSON was scored as a refusal. generator.py retries once with three times this.
    max_answer_tokens: int = 2048

    # Which backend answered, and which prompt did it. Both go on every log
    # line and every span, because "the answer got worse last Tuesday" is
    # unanswerable without them (12.6 puts them in BigQuery).
    # Module 11: THE BACKEND IS A SETTING. vertex = google.genai (the lane); gateway = 11.3's LiteLLM gateway at
    # LITELLM_URL, where GENERATOR_MODEL names a route (documind-slm is 11.4's self-hosted model). Until Module 11
    # this field was reported on every usage row and switched nothing.
    model_backend: str = Field("vertex", alias="MODEL_BACKEND")          # vertex | gateway
    litellm_url: str = Field("", alias="LITELLM_URL")
    gateway_timeout_s: float = Field(90.0, alias="GATEWAY_TIMEOUT_S")   # a cold GPU behind the gateway takes a while
    # 12.6: Model Armor on both sides of the model, behind a switch. Off by default - the lane does not move; a
    # candidate revision with ARMOR=on is where 12.6 judges it. The template is regional (asia-south1, with the data
    # it inspects); guard.py reads the location and the template name from the same variables at import.
    armor: str = Field("off", alias="ARMOR")                              # off | on
    armor_location: str = Field("asia-south1", alias="ARMOR_LOCATION")
    armor_template: str = Field("documind-guard", alias="ARMOR_TEMPLATE")
    # 12.6's answer cache (semantic_cache.py), wired 12 September 2026: off | on. On, a near-enough earlier question
    # of the same tenant under the same corpus fingerprint is answered from Firestore - no retrieval, no model call.
    # Off on the lane until the threshold is measured on paraphrase pairs (the RAG plan, W4).
    semantic_cache: str = Field("off", alias="SEMANTIC_CACHE")
    prompt_id: str = "documind-rag"
    prompt_version: str = "v3"
    retrieval_mode: str = "dense"          # dense | hybrid (4.5's hybrid.py)
    # 4.6's graph, on the lane (13 September 2026; shared/documind_graph.py): off | on | auto. `on` walks the tenant's
    # knowledge graph for every question and puts the chunks its nodes point at in front of the dense pool; `auto`
    # walks it only when the question is relational AND a seed entity is found (4.6's choose_mode, no model call).
    # The graph is built by make graph TENANT= (services/ingest/graph.py); a tenant with no graph is a dense answer,
    # never an error. Judged on a candidate first, like every switch: make candidate RETRIEVAL_GRAPH=auto.
    retrieval_graph: str = Field("off", alias="RETRIEVAL_GRAPH")
    graph_hops: int = Field(1, alias="GRAPH_HOPS")          # 1 or 2: deeper walks return the whole tenant (4.6)
    graph_cap: int = Field(20, alias="GRAPH_CAP")           # nodes per walk, the budget 4.5 defends
    # Where the graph lives (16 September 2026): firestore (graph_nodes / graph_edges beside the chunks, seeded by a
    # name the question contains) or spanner (spanner.tf's DocuMindGraph, seeded BY MEANING: the question's embedding
    # against the names' - GRAPH_SEED_K nearest, none farther than GRAPH_SEED_DISTANCE in cosine distance, so a
    # question about nothing in the graph seeds nothing and `auto` stays dense). make graph GRAPH_BACKEND= builds either.
    graph_backend: str = Field("firestore", alias="GRAPH_BACKEND")
    spanner_instance: str = Field("documind-graph", alias="SPANNER_INSTANCE")
    spanner_database: str = Field("documind", alias="SPANNER_DATABASE")
    graph_seed_k: int = Field(5, alias="GRAPH_SEED_K")
    graph_seed_distance: float = Field(0.4, alias="GRAPH_SEED_DISTANCE")   # unverified on the real corpus: judge it on a candidate

    # USD per 1M tokens, gemini-3.6-flash standard. 12.6 moves this to a
    # BigQuery model_prices table so a rate change is not a redeploy.
    price_in: float = 1.50
    price_out: float = 7.50

    @model_validator(mode="after")
    def _retrieval_modes_agree(self):
        # Refused here, at import, so a revision with an impossible pair never serves: the deploy fails with the
        # message above instead of a service that runs dense and reports hybrid. /version reports the mode that
        # passed this check - the effective one.
        check_retrieval_modes(self.retrieval_backend, self.retrieval_mode, self.retrieval_graph, self.graph_backend)
        return self

settings = Settings()
'''
with open('config.py', 'w') as f: f.write(CONFIG_PY)
print('config.py written')


In [ ]:
DOCUMIND_SCHEMAS_PY = '''
"""The DocuMind answer contract. One definition, imported everywhere. Gap G1.

Until 2026-09-05 there were THREE shapes of "a cited answer" in this course:

    3.2      Citation(chunk_id: int, quote)                      + RAGAnswer
    4.2      Citation(source_id: int, chunk_id: str, relevance)  + RAGResponse(..., needs_more_context)
    rag-api  Citation(chunk_id: str, source_uri, page, quote, score) + RAGAnswer(+model, tokens, latency)

and the plan's gate for Module 3 - "RAGAnswer is byte-identical in 3.2, 4.2 and rag-api" - was
not met. Worse, 9.6's multimodal fields (kind, media_url, start, end) had nowhere to live: the
shared tool projected them and FastAPI stripped them at the response model.

THREE THINGS, NOT ONE, because they are three different moments:

    ModelDraft   what Gemini is ASKED FOR. It cites by [Source N] index, because that is all it
                 can see. It never knows a chunk id, a page, or a score.
    Citation     what a CALLER receives. Resolved from the draft against the chunks the model
                 actually saw, so it carries the id, the source, the page, the score - and,
                 for a figure or a video segment, where to look.
    RAGAnswer    the contract every module hands to the next: answer, citations, confidence,
                 answerable. Nothing about tokens or latency - that is a transport envelope,
                 and rag-api adds it in its own RAGResponse subclass.

3.2 and 4.2 reproduce the three classes below VERBATIM (their notebooks say so), which is what
"byte-identical" means in practice: the text is the same, and the tests in 3.2 import it.
"""
# No `from __future__ import annotations`, on purpose. 3.2 and 4.2 exec these classes as
# a notebook cell, where a deferred `List` never resolves and pydantic refuses to build
# the model. Plain annotations work everywhere the text is pasted, which is the point.
from typing import List, Literal, Optional

from pydantic import BaseModel, Field


class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)
'''

with open('documind_schemas.py', 'w') as f: f.write(DOCUMIND_SCHEMAS_PY)
print('documind_schemas.py:', len(DOCUMIND_SCHEMAS_PY.splitlines()), 'lines')


In [ ]:
SCHEMAS_PY = r'''
from typing import Literal, Optional
from pydantic import BaseModel, Field

# THE contract lives in shared/ - one definition for 3.2, 4.2, this service and every agent.
# Gap G1: until 2026-09-05 this file carried its own Citation/RAGAnswer, the third of three.
from shared.documind_schemas import Citation, DraftCitation, ModelDraft, RAGAnswer, resolve  # noqa: F401

# The filter keys a caller may send (12 September 2026): each one is a restrict namespace the indexer writes on
# the datapoint AND a field on the Firestore row, so the same predicate holds on the dense path, the hybrid path
# and the Firestore fallback alike. tenant_id is the roster's and `current` is the ledger's - never the body's: a
# caller naming either is a header in disguise. main.py refuses any other key with a 400 (check_filters).
FILTER_KEYS = ("doc_type", "kind")

class QueryRequest(BaseModel):
    query: str = Field(min_length=1, max_length=4000)
    tenant_id: str = Field(min_length=1)
    # Optional, and unread: the caller's identity is the verified assertion or bearer token
    # (auth.py), never a body field - a user named in the body is a header in disguise.
    # It stays accepted because Module 4's notebooks still send one; required, it refused the
    # UI, which rightly sends none, with a 422 on the first live sign-in.
    user_id: Optional[str] = None
    top_k: int = Field(default=5, ge=1, le=20)
    stream: bool = True
    filters: Optional[dict] = None  # e.g. {"doc_type": "policy"}: keys from FILTER_KEYS, string values
    # Which harness is asking (8.7, gap G6): the chat service's brains and the UI label
    # themselves so usage_row - and therefore tenant_daily - can compare them. A label only;
    # nothing in retrieval or generation reads it. "mcp" is the agent surface (7.1-7.2): its
    # first live call was a 422, because a new surface has to be added to this list - the
    # list is closed on purpose, so an unknown label is a typo and not a new row in the warehouse.
    brain: Optional[Literal["langchain", "langgraph", "adk", "direct", "ui", "mcp"]] = None

class RAGResponse(RAGAnswer):
    """The contract plus the transport envelope. RAGAnswer is what every module passes along;
    model/tokens/latency are what THIS service knows about the call, and they do not belong
    in a schema 3.2 asks Gemini to fill."""
    model: str
    # Module 11: which backend answered (vertex | gateway) and, from the gateway, what it priced the answer at.
    backend: str = "vertex"
    cost_usd: Optional[float] = None
    tokens_in: int
    tokens_out: int
    # 10.2's context cache, on the answer (12 September 2026): the prompt tokens the model served from it, INSIDE
    # tokens_in and priced at the cache rate (cost.py). Every paid attempt is summed here - the truncation retry's
    # first attempt included - so the row's cost_usd is what was billed.
    cached_tokens: int = 0
    latency_ms: int
    # Where the time went (main.py stage()): retrieve_ms, rerank_ms, generate_ms and the pool the reranker saw.
    # The usage row carries the same four flat, for tenant_daily; here they ride together, for the caller.
    stages: dict[str, int | str] = Field(default_factory=dict)
    # 12.6's answer cache: "semantic" when this answer was served from it (backend=cache, cost 0), else "none".
    cache_hit: Literal["none", "semantic"] = "none"

class StreamEvent(BaseModel):
    # Server-Sent Events payload
    event: Literal["token", "citation", "done", "error"]
    data: dict
'''
with open('schemas.py', 'w') as f: f.write(SCHEMAS_PY)
print('schemas.py written')


In [ ]:
RETRIEVER_PY = '''
import hashlib, json, logging, re
from functools import lru_cache
from google.cloud import aiplatform
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import Namespace
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.vector import Vector
from google.cloud import discoveryengine_v1 as discoveryengine
from google import genai
from google.genai import types
from google.cloud import firestore
from config import settings
from shared.documind_graph import FirestoreGraph, SpannerGraph, choose_mode   # 4.6's stores, the lane's copies (13 and 16 September 2026)

@lru_cache(maxsize=1)
def _genai_client():
    return genai.Client(enterprise=True, project=settings.project_id, location=settings.region)

@lru_cache(maxsize=1)
def _index_endpoint():
    return aiplatform.MatchingEngineIndexEndpoint(settings.vector_index_endpoint)

@lru_cache(maxsize=1)
def _spanner_db():
    """spanner.tf's database, when GRAPH_BACKEND=spanner (16 September 2026); imported here so a Firestore-graph revision
    never loads the Spanner client."""
    from google.cloud import spanner
    return spanner.Client(project=settings.project_id).instance(settings.spanner_instance).database(settings.spanner_database)

def _graph_store():
    return SpannerGraph(_spanner_db()) if settings.graph_backend == "spanner" else FirestoreGraph(_fs())

def embed_for_graph(q: str) -> list[float]:
    """The question as the graph's vectors were made (SEMANTIC_SIMILARITY, the same model): graph.py embeds each
    canonical node name this way, so the distance seed_by_vector ranks by compares like with like. One extra embedding
    call per question, only on the Spanner graph and only when RETRIEVAL_GRAPH is not off."""
    resp = _genai_client().models.embed_content(
        model=settings.embed_model, contents=q,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", output_dimensionality=768))
    return resp.embeddings[0].values

@lru_cache(maxsize=1)
def _fs():
    return firestore.Client(project=settings.project_id, database="(default)")

def embed_query(q: str) -> list[float]:
    # settings.embed_model is EMBEDDING_MODEL in the environment - the SAME variable the ingest worker stamps on
    # every row (variables.tf: embedding_model). Query and document vectors come from one declared model.
    resp = _genai_client().models.embed_content(
        model=settings.embed_model, contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768))
    return resp.embeddings[0].values

def _firestore_fallback(vec: list[float], tenant_id: str, top_k: int, filters: dict | None = None) -> list[dict]:
    """Answer from Firestore when Vector Search will not.

    indexer.py mirrors every embedding here as a Vector field precisely so this
    path exists. It is slower and it skips the ANN tier, but a slower answer is
    a different thing from an outage - and this is the rung the chaos drill
    pulls: undeploy the index, ask a question, get an answer anyway.

    The SAME predicates as the index query (12 September 2026, R06): the tenant, the
    ledger's `current`, and the caller's filters as equality pre-filters on the row's own
    fields. Until then this path had no `filters` parameter, so a doc_type-filtered
    question answered from here read the whole tenant.

    Needs the composite index in 12.5's firestore_indexes.tf - one per predicate
    combination. Without it Firestore does not degrade, it refuses.
    """
    query = _fs().collection(settings.chunks_collection).where("tenant_id", "==", tenant_id)
    if settings.retrieval_current_only == "on":
        # The ledger's promise (12.5): one current version per document. The pre-filter needs the second
        # vector index in firestore_indexes.tf (tenant_id, current, embedding).
        query = query.where("current", "==", True)
    for k, v in (filters or {}).items():
        query = query.where(k, "==", v)       # doc_type, kind: the keys schemas.FILTER_KEYS allows, main.py checked
    hits = (query
            .find_nearest("embedding", Vector(vec),
                          distance_measure=DistanceMeasure.COSINE,
                          limit=top_k,
                          distance_result_field="d").get())
    out = []
    for h in hits:
        d = h.to_dict()
        d["id"] = h.id
        # COSINE distance: smaller is closer, so flip it to a score the
        # reranker can order the same way it orders Vector Search results.
        d["score"] = 1.0 - d.pop("d", 1.0)
        d.pop("embedding", None)          # never ship 768 floats to the model
        # Which rung answered (16 September 2026). The managed backends and the graph have always
        # said; the two rungs of the kit's own tier never did, so a fallback and a hit looked the
        # same to the caller, to tenant_daily, and to anyone watching a demonstration.
        d["found_by"] = "firestore"
        out.append(d)
    return out

def newest_per_source(chunks: list[dict]) -> list[dict]:
    """One version per source, the newest (12 September 2026). The worker swaps a long document in more than one
    batch, so for a moment two versions of one source can both be current, and a candidate set that held both
    would pack both - the reader would be asked to reconcile v1 with v2. Group by source_uri, keep the doc_key
    whose rows landed last (indexed_at, or reactivated_at for the undo), drop the other version's rows. Rows
    without a doc_key or a timestamp (a lane older than the ledger) pass through untouched."""
    newest: dict = {}
    for c in chunks:
        src, key = c.get("source_uri"), c.get("doc_key")
        at = c.get("reactivated_at") or c.get("indexed_at")
        if not (src and key and at is not None):
            continue
        if src not in newest or at > newest[src][1]:
            newest[src] = (key, at)
    return [c for c in chunks
            if not (c.get("doc_key") and c.get("source_uri") in newest and c["doc_key"] != newest[c["source_uri"]][0])]

def prefer_current(chunks: list[dict]) -> list[dict]:
    """Version-chain dedupe, BEFORE the reranker (12.5's ledger). A chunk the ledger has retired is never a
    source, whether or not its successor was retrieved - the model is not asked to reconcile v1 with v2.
    Chunks without the field (a lane older than the ledger) pass through; a no-op until the flag is stamped.
    Then one version per source: the newest-per-source guard closes the swap window on its own."""
    return newest_per_source([c for c in chunks if c.get("current") is not False])

@lru_cache(maxsize=1)
def _rag():
    """vertexai.rag on the corpora's region (P9.4, 13 September 2026): serverless corpora are us-central1-only (4.3),
    and this client never generates - generation stays on the global client the generator holds."""
    import vertexai
    from vertexai import rag
    vertexai.init(project=settings.project_id, location=settings.rag_location)
    return rag

_corpora = {}          # tenant_id -> the corpus's resource name, once found

def _rag_corpus(tenant_id: str) -> str | None:
    """The tenant's corpus, by the mirror's name (documind-{tenant}, services/ingest/managed.py), found once and kept;
    None while the tenant has none - looked up again on the next question, never created here."""
    if tenant_id not in _corpora:
        want = "documind-" + re.sub(r"[^a-z0-9-]+", "-", tenant_id.lower()).strip("-")
        name = next((c.name for c in _rag().list_corpora() if c.display_name == want), None)
        if name is None:
            return None
        _corpora[tenant_id] = name
    return _corpora[tenant_id]

def _version_row(tenant_id: str, doc_key: str) -> dict | None:
    """What a managed context lacks, from the kit's own rows: the version's source_uri, doc_type, effective_from,
    indexed_at - and `current`, read fresh, so a version the ledger retired while the store still held it is dropped
    by prefer_current() like any retired row. One small read per distinct version in the pool."""
    for snap in (_fs().collection(settings.chunks_collection).where("tenant_id", "==", tenant_id)
                 .where("doc_key", "==", doc_key).limit(1).stream()):
        d = snap.to_dict() or {}
        return {k: d.get(k) for k in ("source_uri", "doc_type", "effective_from", "indexed_at", "reactivated_at", "current")}
    return None

def _page_span(ctx) -> str:
    span = getattr(getattr(ctx, "chunk", None), "page_span", None)
    first, last = getattr(span, "first_page", 0) or 0, getattr(span, "last_page", 0) or 0
    return f"p{first}" + (f"-{last}" if last and last != first else "") if first else ""

def _media_rows(vec: list[float], tenant_id: str, top_k: int, filters: dict | None = None) -> list[dict]:
    """The kit's own figure and segment rows (Module 9), for a pool a managed backend served: a store holds text only
    (the plan's D4), so media comes from Firestore's vector index under the tenant + kind index firestore_indexes.tf
    declares. A doc_type filter is applied to the rows returned - the kind index carries no doc_type."""
    query = _fs().collection(settings.chunks_collection).where("tenant_id", "==", tenant_id)
    if settings.retrieval_current_only == "on":
        query = query.where("current", "==", True)
    if filters and filters.get("kind"):
        query = query.where("kind", "==", filters["kind"])          # the caller named the kind: one equality, as the fallback does
    else:
        query = query.where("kind", "in", ["figure", "segment"])     # the media kinds a store never holds
    hits = query.find_nearest("embedding", Vector(vec), distance_measure=DistanceMeasure.COSINE,
                              limit=top_k, distance_result_field="d").get()
    out = []
    for h in hits:
        d = h.to_dict()
        if filters and filters.get("doc_type") and d.get("doc_type") != filters["doc_type"]:
            continue
        d["id"], d["score"] = h.id, 1.0 - d.pop("d", 1.0)
        d.pop("embedding", None)
        out.append(d)
    return out

def _managed_retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None,
                      vec: list[float] | None = None) -> list[dict]:
    """RETRIEVAL_BACKEND=rag_engine (P9.4, 13 September 2026): lesson 4.3's corpus as the kit's retrieval stage, and
    nothing more - the reranker, the packing, the generator and the citations stay the kit's (the plan's D1).

    The tenant's corpus is the mirror's (services/ingest/managed.py: one per tenant, a RagFile per version named by
    its doc_key, the version's own text), queried by text with 4.3's distance threshold. Each context comes back with
    the RagFile's display name - the doc_key - and that is enough to make it a chunk of the kit's contract: the
    version's row gives it source_uri, doc_type, effective_from, indexed_at and a fresh `current`; the id is stable
    (tenant, doc_key, a hash of the text) so the answer cache and the citations hold; the score is 1 - distance,
    the scale the Firestore path uses. A caller's filters apply to the mapped chunks as on every path; figure and
    segment rows join from the kit's own index (D4), unless the caller asked for text alone. A tenant with no
    corpus, or a store that will not answer, is the Firestore rung with the filters, logged rag_engine_fallback -
    never an empty pool that reads as a refusal. A tenant whose data_region is `in` never reaches this: main.py's
    choose_for sends it to the kit's own index (policy_fallback) before retrieve() is called (13 September 2026)."""
    kind = (filters or {}).get("kind")
    if kind and kind != "text":                        # figures or segments only: the store has none, the index has them
        return _media_rows(vec, tenant_id, settings.top_k_retrieve, filters)
    corpus = _rag_corpus(tenant_id)
    if corpus is None:
        logging.warning(json.dumps({"event": "rag_engine_fallback", "tenant": tenant_id,
                                    "reason": f"no corpus for the tenant: make rag-corpus TENANT={tenant_id}"}))
        return _firestore_fallback(vec, tenant_id, settings.top_k_retrieve, filters)
    try:
        rag = _rag()
        resp = rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=corpus)], text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=settings.top_k_retrieve,
                filter=rag.Filter(vector_distance_threshold=settings.rag_distance_threshold)))
        contexts = list(resp.contexts.contexts)
    except Exception as e:
        logging.warning(json.dumps({"event": "rag_engine_fallback", "tenant": tenant_id, "error": str(e)[:200]}))
        return _firestore_fallback(vec, tenant_id, settings.top_k_retrieve, filters)
    rows: dict[str, dict | None] = {}
    out = []
    for ctx in contexts:
        doc_key, text = getattr(ctx, "source_display_name", "") or "", getattr(ctx, "text", "") or ""
        if not doc_key or not text.strip():
            continue
        if doc_key not in rows:
            rows[doc_key] = _version_row(tenant_id, doc_key)
        row = rows[doc_key]
        if row is None:                                # a file the ledger does not know: never served
            continue
        chunk = {"id": f"{tenant_id}:{doc_key}#rag-{hashlib.sha256(text.encode('utf-8')).hexdigest()[:12]}",
                 "text": text, "source_uri": row.get("source_uri") or "", "doc_key": doc_key,
                 "doc_type": row.get("doc_type"), "kind": "text", "effective_from": row.get("effective_from"),
                 "indexed_at": row.get("indexed_at"), "reactivated_at": row.get("reactivated_at"),
                 "current": row.get("current"), "locator": _page_span(ctx),
                 "score": max(0.0, 1.0 - float(getattr(ctx, "score", 0.0) or 0.0)), "found_by": "rag_engine"}
        if any(chunk.get(k) != v for k, v in (filters or {}).items()):
            continue
        out.append(chunk)
    if kind != "text":
        out += _media_rows(vec, tenant_id, max(3, settings.top_k_retrieve // 4), filters)
    return sorted(out, key=lambda c: -c["score"])[:settings.top_k_retrieve]

@lru_cache(maxsize=1)
def _search():
    """The Vertex AI Search client (R4, 13 September 2026 evening): one per process. The data stores are global
    (managed.tf), so no region is chosen here; the client's default endpoint serves them."""
    return discoveryengine.SearchServiceClient()

def _search_serving_config(tenant_id: str) -> str:
    """The tenant's data store, by the mirror's id (documind-{tenant}: services/ingest/managed.py's store_id, the same
    regex), searched through its default serving config the way 4.4's notebook searches it - no engine needed."""
    store = "documind-" + re.sub(r"[^a-z0-9-]+", "-", tenant_id.lower()).strip("-")
    return (f"projects/{settings.project_id}/locations/{settings.search_location}/collections/default_collection"
            f"/dataStores/{store}/servingConfigs/default_search")

def _search_filter(filters: dict | None) -> str:
    """The caller's filters as a Vertex AI Search filter expression on the schema's indexable fields (managed.tf):
    doc_type: ANY("policy"). `kind` never reaches the store - it holds text only (D4): a media kind is answered from
    the kit's index before the store is asked, and text is what every document there is."""
    return " AND ".join(f'{k}: ANY("{str(v).replace(chr(34), chr(92) + chr(34))}")' for k, v in (filters or {}).items() if k != "kind")

def _search_texts(doc) -> list[tuple[str, str]]:
    """What the store extracted for one result: its extractive segments (content, pageNumber) when it serves them, else
    its snippet with the markup stripped - the field 4.4's cell reads; nothing for a document with neither."""
    dd = ((discoveryengine.Document.to_dict(doc).get("derived_struct_data") or {}) if hasattr(discoveryengine.Document, "to_dict")
          else dict(getattr(doc, "derived_struct_data", None) or {}))
    out = []
    for seg in dd.get("extractive_segments") or []:
        text = str(seg.get("content") or "").strip()
        if text:
            out.append((text, str(seg.get("pageNumber") or "")))
    if not out:
        for snip in dd.get("snippets") or []:
            text = re.sub(r"<[^>]+>", "", str(snip.get("snippet") or "")).strip()
            if text:
                out.append((text, ""))
    return out

def _search_retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None,
                     vec: list[float] | None = None) -> list[dict]:
    """RETRIEVAL_BACKEND=vertex_search (R4, 13 September 2026 evening): lesson 4.4's data store as the kit's retrieval
    stage, to the same contract as rag_engine (the plan's D1) - the store ranks, the kit reranks, packs, generates
    and cites.

    The tenant's data store is the mirror's (managed.tf; services/ingest/managed.py: one Document per current version,
    its id the doc_key, its content the version's text), searched by text through its default serving config as
    4.4's notebook searches it. Each result is the version's text as the store extracted it - extractive segments
    when it serves them, else the snippet - one chunk each, mapped to the kit's chunk contract through the version's
    own row (source_uri, doc_type, effective_from, a fresh `current`); the id is stable (tenant, doc_key, a hash of
    the text), the score is the rank (Vertex AI Search orders, it does not score: 1.0, 0.99, ... until the reranker
    orders the pool), found_by vertex_search. The caller's doc_type goes to the store as a filter expression on its
    structData and is applied once more to the mapped chunks; figures and segments join from the kit's own index
    (D4) unless the caller asked for text alone. No results is an empty pool - the store answered. A tenant with no
    data store, or a store that will not answer, is the Firestore rung with the filters, logged vertex_search_fallback."""
    kind = (filters or {}).get("kind")
    if kind and kind != "text":                        # figures or segments only: the store has none, the index has them
        return _media_rows(vec, tenant_id, settings.top_k_retrieve, filters)
    try:
        spec = discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(return_snippet=True),
            extractive_content_spec=discoveryengine.SearchRequest.ContentSearchSpec.ExtractiveContentSpec(
                max_extractive_segment_count=settings.search_segments))
        results = list(_search().search(request=discoveryengine.SearchRequest(
            serving_config=_search_serving_config(tenant_id), query=query, page_size=settings.top_k_retrieve,
            filter=_search_filter(filters), content_search_spec=spec)))
    except Exception as e:
        logging.warning(json.dumps({"event": "vertex_search_fallback", "tenant": tenant_id, "error": str(e)[:200],
                                    "hint": "no data store for the tenant (MANAGED_SEARCH=true make up declares one per `any` tenant), or the store did not answer"}))
        return _firestore_fallback(vec, tenant_id, settings.top_k_retrieve, filters)
    rows: dict[str, dict | None] = {}
    out, rank = [], 0
    for r in results:
        doc = getattr(r, "document", None)
        doc_key = (getattr(doc, "id", "") or "") if doc is not None else ""
        if not doc_key:
            continue
        if doc_key not in rows:
            rows[doc_key] = _version_row(tenant_id, doc_key)
        row = rows[doc_key]
        if row is None:                                # a document the ledger does not know: never served
            continue
        for text, page in _search_texts(doc):
            chunk = {"id": f"{tenant_id}:{doc_key}#vs-{hashlib.sha256(text.encode('utf-8')).hexdigest()[:12]}",
                     "text": text, "source_uri": row.get("source_uri") or "", "doc_key": doc_key,
                     "doc_type": row.get("doc_type"), "kind": "text", "effective_from": row.get("effective_from"),
                     "indexed_at": row.get("indexed_at"), "reactivated_at": row.get("reactivated_at"),
                     "current": row.get("current"), "locator": f"p{page}" if page else "",
                     "score": max(0.0, 1.0 - rank / 100), "found_by": "vertex_search"}
            rank += 1
            if any(chunk.get(k) != v for k, v in (filters or {}).items()):
                continue
            out.append(chunk)
    if kind != "text":
        out += _media_rows(vec, tenant_id, max(3, settings.top_k_retrieve // 4), filters)
    return sorted(out, key=lambda c: -c["score"])[:settings.top_k_retrieve]

def _dense_retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None,
                    vec: list[float] | None = None, backend: str | None = None) -> list[dict]:
    """The dense pool: Vector Search (dense or hybrid) with the Firestore fallback beneath it, Firestore's own
    vector index when a request is routed to it, or a managed store (rag_engine, P9.4; vertex_search, R4) - the same tenant /
    current / filter predicates on every path. `backend` is the one main.py chose for THIS request (the tenant's pin, held against
    its data_region - 13 September 2026, evening); the deployment's RETRIEVAL_BACKEND when the caller names none."""
    backend = backend or settings.retrieval_backend
    vec = vec if vec is not None else embed_query(query)    # main.py embeds once: the answer cache looked it up first
    if backend == "rag_engine":
        return prefer_current(_managed_retrieve(query, tenant_id, top_k, filters, vec=vec))
    if backend == "vertex_search":
        return prefer_current(_search_retrieve(query, tenant_id, top_k, filters, vec=vec))
    if backend == "firestore":
        # The Firestore rung on its own (RETRIEVAL_BACKEND=firestore, or a tenant pinned to it): no endpoint is asked.
        # Firestore holds every embedding indexer.py wrote and its own vector index answers,
        # tenant pre-filtered - the fallback below, chosen rather than fallen into.
        return prefer_current(_firestore_fallback(vec, tenant_id, settings.top_k_retrieve, filters))
    restricts = [Namespace(name="tenant_id", allow_tokens=[tenant_id])]
    if settings.retrieval_current_only == "on":
        restricts.append(Namespace(name="current", allow_tokens=["true"]))   # indexer.py's third restrict
    if filters:
        for k, v in filters.items():
            restricts.append(Namespace(name=k, allow_tokens=[str(v)]))
    try:
        if settings.retrieval_mode == "hybrid":
            # 4.5's hybrid.py, wired. Dense recall misses exact tokens - an
            # invoice number, a clause id - and sparse misses paraphrase. RRF
            # over both is what 4.5 measured; this is where it earns its keep.
            # The SAME restricts as the dense query (12 September 2026): a filter is a
            # predicate on the corpus, not on one of the two ways through it.
            from hybrid import hybrid_find_neighbors
            neighbours = hybrid_find_neighbors(
                _index_endpoint(), settings.vector_deployed_index, vec,
                query, tenant_id, settings.top_k_retrieve, alpha=0.7, restricts=restricts)
        else:
            resp = _index_endpoint().find_neighbors(
                deployed_index_id=settings.vector_deployed_index,
                queries=[vec], num_neighbors=settings.top_k_retrieve,
                filter=restricts,
            )
            # One query in, one neighbour list out - and an EMPTY outer list when the index
            # holds nothing for these restricts. That is an empty pool (main.py answers it
            # without a model call), not an outage: it must not fall through to the except
            # below and come back from Firestore as if the index were down (12 September 2026).
            neighbours = resp[0] if resp else []
    except Exception as e:
        # The chaos rung. An undeployed or unreachable index raises here;
        # Firestore holds the same vectors, so answer from there and say so
        # in the log rather than returning nothing. Hybrid takes the same rung:
        # an outage degrades to dense, logged - it never 500s.
        logging.warning(json.dumps({"event": "vector_search_fallback",
                                    "tenant": tenant_id, "error": str(e)[:200]}))
        return prefer_current(_firestore_fallback(vec, tenant_id, settings.top_k_retrieve, filters))
    ids = [n.id for n in neighbours]
    scores = {n.id: n.distance for n in neighbours}
    # Fan-out to Firestore for chunk payloads. ALL of them: we asked Vector
    # Search for top_k_retrieve (20) and then used to fetch ids[:10], so half
    # of every retrieval was thrown away before the reranker ever saw it.
    # Firestore's `in` takes up to 30 values: 20 is one query, and a pool of 50
    # (TOP_K_RETRIEVE, the knob the ablation moves) is two - never one query
    # over the limit, which Firestore refuses rather than truncates.
    pool = _hydrate(ids[:settings.top_k_retrieve], scores)
    for c in pool:
        c["found_by"] = "vector"          # these ids came from find_neighbors, not from Firestore's own index (16 September 2026)
    return prefer_current(pool)


def graph_candidates(query: str, tenant_id: str, filters: dict | None = None) -> list[dict]:
    """4.6's graph, on the lane (13 September 2026): the chunks the tenant's knowledge graph points at for this question.

    RETRIEVAL_GRAPH=off returns nothing and touches nothing. `on` walks for every question; `auto` walks only when the
    question is relational and a seed entity is found (shared/documind_graph.choose_mode - no model call). The walk is
    the notebook's own class (FirestoreGraph: seed by containment, expand one or two hops, capped) and it is
    tenant-scoped in every read; the chunk ids it hands over are fetched from the one chunks collection and checked
    once more - the tenant, the caller's filters, the ledger's current - the same predicates every other path applies.
    A tenant with no graph, or a question with no seed, is an empty list: dense retrieval answers as before."""
    mode = settings.retrieval_graph
    if mode == "off":
        return []
    g = _graph_store()
    if settings.graph_backend == "spanner":
        # By meaning (16 September 2026): the nearest node names to the question, none farther than the threshold.
        # choose_mode still asks whether the question is relational, so `auto` stays dense for a plain lookup.
        seeds = g.seed_by_vector(embed_for_graph(query), tenant_id, k=settings.graph_seed_k, max_distance=settings.graph_seed_distance)
    else:
        seeds = g.seed(query, tenant_id)
    if not seeds or (mode == "auto" and choose_mode(query, seeds) != "graph"):
        return []
    nodes = g.expand([s["node_id"] for s in seeds], tenant_id, hops=settings.graph_hops, cap=settings.graph_cap)
    out = []
    for cid in sorted({c for n in nodes for c in n["chunk_ids"]}):
        snap = _fs().collection(settings.chunks_collection).document(cid).get()
        if not snap.exists:
            continue
        d = snap.to_dict() or {}
        if d.get("tenant_id") != tenant_id:            # the walk is tenant-scoped; the id it hands over is checked once more
            continue
        if any(d.get(k) != v for k, v in (filters or {}).items()):   # doc_type, kind: the same predicates as every path
            continue
        if settings.retrieval_current_only == "on" and d.get("current") is not True:
            continue
        d["id"] = cid
        d["score"] = 1.0                               # fetched by id: an exact hit, ahead of the dense pool
        d["found_by"] = "graph"
        d.pop("embedding", None)
        out.append(d)
    return out


def retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None,
             vec: list[float] | None = None, backend: str | None = None) -> list[dict]:
    """The pool the reranker sees: the graph's chunks first, when RETRIEVAL_GRAPH says so, then the dense candidates
    that are not already in it, cut to TOP_K_RETRIEVE; a retired version never survives either half. `backend` is
    the request's (main.py), the setting by default."""
    graph = graph_candidates(query, tenant_id, filters)
    dense = _dense_retrieve(query, tenant_id, top_k, filters, vec=vec, backend=backend)
    if not graph:
        return dense
    seen = {c["id"] for c in graph}
    return prefer_current(graph + [c for c in dense if c["id"] not in seen])[:settings.top_k_retrieve]


def _hydrate(ids: list[str], scores: dict) -> list[dict]:
    """The chunk payloads for Vector Search's ids, in Vector Search's order, 30 ids per `in` query; nothing for none."""
    by_id: dict[str, dict] = {}
    for start in range(0, len(ids), 30):
        for doc in _fs().collection(settings.chunks_collection).where(
                "__name__", "in", ids[start:start + 30]).stream():
            d = doc.to_dict(); d["id"] = doc.id; d["score"] = scores.get(doc.id, 0)
            by_id[doc.id] = d
    return [by_id[i] for i in ids if i in by_id]

@lru_cache(maxsize=1)
def _ranker():
    return discoveryengine.RankServiceClient()

def _by_retrieval_score(chunks: list[dict], k: int) -> list[dict]:
    """The pool by retrieval score, cut to k, every row marked - what rerank() returns when the Ranking API
    cannot answer. `score` is a similarity on both backends (DOT_PRODUCT on the index, vector.tf; 1 - cosine
    on the Firestore fallback): highest first, ties in the order retrieval gave."""
    out = sorted(chunks, key=lambda c: -(c.get("score") or 0.0))[:k]
    for c in out:
        c["rerank_fallback"] = True
    return out


def rerank_fell_back(chunks: list[dict]) -> bool:
    """True when rerank() stood the pool in for the Ranking API: the handlers put stages['rerank_fallback'] = 1 on
    the row and in the answer, so a day of fallbacks is a count, not a hunch about worse answers."""
    return any(c.get("rerank_fallback") for c in chunks)


def rerank(query: str, chunks: list[dict], k: int, tenant_id: str | None = None) -> list[dict]:
    """The Ranking API's order, top k - or, when it does not answer inside RERANK_TIMEOUT_S or raises at all,
    the pool by retrieval score, logged as rerank_fallback (12 September 2026, R06). Before this a slow or absent
    ranker was a hung request and then a 500; a worse order is a degraded answer, and the row says so."""
    if not chunks: return chunks
    chunks = chunks[:200]          # the Ranking API takes at most 200 records; the pool is TOP_K_RETRIEVE, never more
    try:
        client = _ranker()
        ranking_config = client.ranking_config_path(
            project=settings.project_id, location="global",
            ranking_config="default_ranking_config")
        records = [discoveryengine.RankingRecord(id=str(i), content=c["text"])
                   for i, c in enumerate(chunks)]
        resp = client.rank(request=discoveryengine.RankRequest(
            ranking_config=ranking_config,
            model=settings.rerank_model,
            top_n=k, query=query, records=records),
            timeout=settings.rerank_timeout_s)      # gapic's deadline for the whole call, retries included
    except Exception as e:  # noqa: BLE001 - a deadline, a quota, a client that will not build: one answer
        logging.warning(json.dumps({"event": "rerank_fallback", "tenant": tenant_id,
                                    "error": type(e).__name__}))
        return _by_retrieval_score(chunks, k)
    out = []
    for r in resp.records:
        chunks[int(r.id)]["rerank_score"] = r.score
        out.append(chunks[int(r.id)])
    return out
'''
with open('retriever.py', 'w') as f: f.write(RETRIEVER_PY)
print('retriever.py written')


In [ ]:
GENERATOR_PY = '''
import json
import logging
import re
import time
from types import SimpleNamespace

import httpx
from fastapi import HTTPException
from pydantic import ValidationError

from google import genai
from google.genai import errors, types
from schemas import DraftCitation, ModelDraft, RAGResponse, resolve
from config import settings
from context_budget import pack_chunks, estimate_tokens, TokenBudget
from cache_manager import TenantCacheManager
from functools import lru_cache


@lru_cache(maxsize=1)
def _caches() -> TenantCacheManager:
    return TenantCacheManager(settings.project_id, settings.region)

log = logging.getLogger("documind-api")   # same logger main.py writes usage rows on
_client = genai.Client(enterprise=True, project=settings.project_id, location="global")  # generation runs on the global endpoint (Gemini 3.x)


@lru_cache(maxsize=8)
def _client_at(location: str) -> genai.Client:
    return _client if location == "global" else genai.Client(enterprise=True, project=settings.project_id, location=location)


def _regional_client() -> genai.Client:
    return _client_at(settings.region)


def _endpoint_location(model: str) -> str:
    """Where a tuned endpoint is served from: GENERATOR_LOCATION when set, else the path's own
    /locations/<x>/ segment, else the service's region. The first live tuning job (10 September) put its
    endpoint in the `us` multi-region while the service assumed us-central1 (F41): the path knows, the
    service does not, and the setting overrides both without a code change."""
    if settings.generator_location:
        return settings.generator_location
    m = re.search(r"/locations/([^/]+)/", model)
    return m.group(1) if m else settings.region


def _client_for(model: str) -> genai.Client:
    """A model NAME is served on the global endpoint; a TUNED model is an endpoint path, served from the
    location the path names (10.1: tuning is regional; the endpoint may land in a multi-region) and the
    global client answers 404 for it. The value of GENERATOR_MODEL decides - so a tuned model behind the
    same retrieve() is a redeploy with one variable, and every surface above the API (MCP, the agents,
    the UI) changes nothing."""
    return _client_at(_endpoint_location(model)) if model.startswith("projects/") else _client


# ----------------------------------------------------------------------------- the gateway backend (Module 11)
# THE BACKEND IS A SETTING. MODEL_BACKEND=vertex is what the lane runs: google.genai, the model above. MODEL_BACKEND=gateway
# sends the SAME prompt - SYSTEM, the packed context, the question - to the LiteLLM gateway (11.3) as an OpenAI-compatible
# chat completion with a JSON response format, and reads the SAME ModelDraft back through the same resolve(). With the
# gateway, GENERATOR_MODEL names a ROUTE (documind-general, documind-slm, documind-inference, documind-gke), so 11.4's
# self-hosted model answers behind the same retrieve() as Gemini, and every surface above the API changes nothing. The
# gateway is a Cloud Run service behind IAM like every other: this service's own account mints the ID token.
GATEWAY_JSON_RULE = ('\\nReply with JSON only, exactly this shape: {"answer": str, "citations": [{"source": int, "quote": str}], '
                     '"confidence": "high" | "medium" | "low", "answerable": bool}.')
GATEWAY_ROUTES = ("documind-general", "documind-reasoning", "documind-slm", "documind-inference", "documind-gke")
_gateway_token = {"token": None, "exp": 0.0}


def _gateway_url() -> str:
    if not settings.litellm_url:
        raise HTTPException(502, "MODEL_BACKEND=gateway but LITELLM_URL is not set")
    return settings.litellm_url.rstrip("/")


def _gateway_headers() -> dict:
    """An ID token for the gateway's audience, minted by this service's own account through the metadata server (the
    same leg documind_tools uses on Cloud Run), cached and refreshed five minutes early."""
    if not _gateway_token["token"] or time.time() > _gateway_token["exp"] - 300:
        import google.auth.transport.requests
        import google.oauth2.id_token
        _gateway_token["token"] = google.oauth2.id_token.fetch_id_token(google.auth.transport.requests.Request(), _gateway_url())
        _gateway_token["exp"] = time.time() + 3600
    return {"Authorization": f"Bearer {_gateway_token['token']}", "Content-Type": "application/json"}


def _gateway_route(model: str) -> str:
    """A Gemini name or a tuned endpoint under the gateway backend means the gateway's Gemini route; a route name is
    passed through. The tuned Gemini endpoint (10.1) is served directly by the vertex backend, never through the gateway."""
    if model.startswith("gemini-3.1-pro"):
        return "documind-reasoning"
    if model.startswith("gemini-") or model.startswith("projects/"):
        return "documind-general"
    return model


def _gateway_messages(prompt: str) -> list[dict]:
    """The prompt _call() sends, split into the OpenAI shape: SYSTEM (plus the JSON rule) as the system turn, the
    context and the question as the user turn. Text only: a self-hosted model reads the caption, never the pixels."""
    user = prompt[len(SYSTEM):].lstrip() if prompt.startswith(SYSTEM) else prompt
    return [{"role": "system", "content": SYSTEM + GATEWAY_JSON_RULE}, {"role": "user", "content": user}]


class _GatewayReply:
    """The attributes _draft() and _finish_reason() read, over an OpenAI-compatible reply from the gateway."""
    parsed = None
    prompt_feedback = None

    def __init__(self, j: dict, cost_usd: float | None, model: str):
        choice = (j.get("choices") or [{}])[0]
        self.text = (choice.get("message") or {}).get("content") or ""
        fr = (choice.get("finish_reason") or "stop")
        self.candidates = [SimpleNamespace(finish_reason="MAX_TOKENS" if fr == "length" else fr.upper())]
        u = j.get("usage") or {}
        self.usage_metadata = SimpleNamespace(prompt_token_count=u.get("prompt_tokens") or 0,
                                              candidates_token_count=u.get("completion_tokens") or 0,
                                              thoughts_token_count=None, cached_content_token_count=0)
        self.model = j.get("model") or model
        self.cost_usd = cost_usd


def _gateway_call(prompt: str, packed: list[dict], tenant_id: str | None, max_tokens: int, model: str) -> _GatewayReply:
    body = {"model": model, "messages": _gateway_messages(prompt), "response_format": {"type": "json_object"},
            "max_tokens": max_tokens, "metadata": {"tenant": tenant_id or ""}}
    try:
        r = httpx.post(f"{_gateway_url()}/v1/chat/completions", json=body, headers=_gateway_headers(),
                       timeout=settings.gateway_timeout_s)
    except httpx.HTTPError as e:
        raise HTTPException(502, f"gateway unreachable: {type(e).__name__}") from e
    if r.status_code != 200:
        raise HTTPException(502, f"gateway {r.status_code}: {r.text[:200]}")
    cost = r.headers.get("x-litellm-response-cost")           # the gateway prices the route it served, fallbacks included
    return _GatewayReply(r.json(), float(cost) if cost else None, model)


def _gateway_stream(prompt: str, packed: list[dict], tenant_id: str | None, model: str):
    """The gateway's SSE, re-yielded as ("token", text) then ("usage", dict) - the same two kinds /v1/stream emits."""
    body = {"model": model, "messages": _gateway_messages(prompt), "stream": True, "stream_options": {"include_usage": True},
            "max_tokens": settings.max_answer_tokens, "metadata": {"tenant": tenant_id or ""}}
    usage = {"tokens_in": 0, "tokens_out": 0, "cached_tokens": 0}
    cost = None
    with httpx.stream("POST", f"{_gateway_url()}/v1/chat/completions", json=body, headers=_gateway_headers(),
                      timeout=settings.gateway_timeout_s) as r:
        if r.status_code != 200:
            raise HTTPException(502, f"gateway {r.status_code}")
        for line in r.iter_lines():
            if not line.startswith("data:"):
                continue
            payload = line[5:].strip()
            if payload == "[DONE]":
                break
            j = json.loads(payload)
            for ch in j.get("choices") or []:
                t = (ch.get("delta") or {}).get("content")
                if t:
                    yield "token", t
            if j.get("usage"):
                usage = {"tokens_in": j["usage"].get("prompt_tokens") or 0,
                         "tokens_out": j["usage"].get("completion_tokens") or 0, "cached_tokens": 0}
        cost = r.headers.get("x-litellm-response-cost")
    yield "usage", {**usage, "model": model, "backend": "gateway", "cost_usd": float(cost) if cost else None}


SYSTEM = """You are DocuMind, a retrieval-grounded assistant.
Rules:
1. Answer ONLY from the numbered context below. Never invent sources.
2. Cite using [N] where N is the chunk number. Multiple chunks: [1,2].
3. If the context does not contain the answer, set answerable=false and say so.
4. Keep answers under 300 words unless asked for more.
5. A quote is the clause that answers - at most twenty-five words, never a whole section.
"""

# The ledger's second half (12.5, 11 September 2026): a document may declare when it applies, and two sources
# in one context may then disagree by date. The rule is added ONLY when a packed chunk carries a date - SYSTEM
# itself stays byte-identical to the tuning dataset's (evals/make_trainset.py), so the tuned model is served
# behind the prompt it was trained behind.
DATED_RULE = ("\\n6. Some sources carry an effective date. Where sources disagree, follow the one with the latest"
              " effective date, cite it, and say which source you followed and from when it applies.")


def _dated_rule(packed: list[dict]) -> str:
    return DATED_RULE if any(c.get("effective_from") for c in packed) else ""


def _budget(query: str, count_fn=estimate_tokens) -> TokenBudget:
    """The request's budget lines (4.5's TokenBudget, imported by nothing until 12 September 2026): the fixed parts
    of the prompt are counted - SYSTEM, the dated rule and the question with its scaffolding - and the chunks get
    what is LEFT of max_context_tokens. Until then the whole of max_context_tokens went to the chunks and the fixed
    parts rode on top, so the prompt exceeded the configured total by their size on every full request (R06). The
    dated rule is reserved whether or not a packed chunk turns out to carry a date: that is known only after
    packing, and holding its room is the safe side. count_fn is estimate_tokens (len // 4), the counter pack_chunks
    uses per block; a real count - client.models.count_tokens - can be injected here and there alike."""
    fixed = f"{SYSTEM}{DATED_RULE}\\n\\nContext:\\n\\n\\nQuestion: {query}"      # the prompt below, with nothing packed
    return TokenBudget.fit(settings.max_context_tokens, fixed, count_fn, answer=settings.max_answer_tokens)


def _pack(query: str, chunks: list[dict]) -> tuple[str, list[dict]]:
    """The context and the packed set for the prompt: pack_chunks inside the chunk budget, the drops logged.
    THREE values from pack_chunks, not one: (context, packed, dropped) - 4.5's contract, and its own docstring
    says so. Assigning the tuple to `chunks` and handing it to build_context() raised AttributeError: 'str'
    object has no attribute 'get' on EVERY request, while /health and /ready both stayed green."""
    context, packed, dropped = pack_chunks(chunks, _budget(query).chunks, estimate_tokens)
    if dropped:
        # Not silent. A dropped chunk is a passage the model was never shown, and
        # "the answer got worse after we added documents" starts here.
        log.info(json.dumps({"event": "context_budget_drop",
                             "packed": len(packed), "dropped": len(dropped)}))
    return context, packed


def _usage(r) -> dict:
    """What one attempt billed (6.2, 12 September 2026): the prompt tokens - the cached ones among them, priced at
    the cache rate by cost.py - and the output, which on the 3.x family is the candidates AND the thinking:
    thoughts_token_count is billed as output whether or not a thought is shown, and until now it was never added,
    so every row under-counted what the month's counter (budget.py) then read."""
    u = getattr(r, "usage_metadata", None)
    n = lambda k: getattr(u, k, None) or 0  # noqa: E731 - None from the SDK means "not this call", not 0 tokens
    return {"tokens_in": n("prompt_token_count"),
            "tokens_out": n("candidates_token_count") + n("thoughts_token_count"),
            "cached_tokens": n("cached_content_token_count")}


def _add(a: dict, b: dict) -> dict:
    """Two attempts' usage as one bill: the tokens summed, the gateway's price summed when either attempt carried one."""
    out = {k: (a.get(k) or 0) + (b.get(k) or 0) for k in ("tokens_in", "tokens_out", "cached_tokens")}
    if a.get("cost_usd") is not None or b.get("cost_usd") is not None:
        out["cost_usd"] = (a.get("cost_usd") or 0.0) + (b.get("cost_usd") or 0.0)
    return out


# build_context() is GONE, deliberately. pack_chunks() already returns a formatted
# context, and keeping both meant two functions formatting the same sources two
# different ways - `[i] src page N` here and `[Source n] ...` in context_budget.py.
# One of them was always going to be the one nobody updated.
def _contents(prompt: str, packed: list[dict]) -> list:
    """9.6 cell 15, shipped: show the model the figure it is about to cite.

    A verbalised caption is enough to RETRIEVE a figure; it is not always enough to ANSWER
    from one - "which segment grew fastest" needs the chart, not the sentence about the chart.
    So the image Part is appended for every packed chunk that is a figure or a table and has a
    locator. Only those: each crop is ~258+ tokens, and most questions are answered by the
    caption alone. from_uri: the asset stays in GCS and never passes through this process.
    """
    contents: list = [prompt]
    for i, c in enumerate(packed, 1):
        if c.get("kind") in ("figure", "table") and c.get("media_url"):
            mime = "image/jpeg" if str(c["media_url"]).lower().endswith((".jpg", ".jpeg")) else "image/png"
            contents.append(types.Part.from_uri(file_uri=c["media_url"], mime_type=mime))
            contents.append(f"(the image above is [Source {i}])")
    return contents


def _cache_kwargs(tenant_id: str | None, model: str | None = None) -> dict:
    """4.5's cache_manager, wired. Empty dict when the tenant has no live cache - or when the cache
    was made for another model (10.2: a cache is the model's, not the tenant's alone)."""
    if not tenant_id:
        return {}
    try:
        return _caches().generate_config_kwargs(tenant_id, model)
    except Exception:
        # A cache miss must never fail the answer. Caching is an optimisation,
        # and an optimisation that can take the service down is a liability.
        return {}


def _call(prompt: str, packed: list[dict], tenant_id: str | None, max_tokens: int, model: str | None = None):
    """One generate_content call with the structured-answer config, on the client the model needs."""
    model = model or settings.generator_model
    return _client_for(model).models.generate_content(
        model=model,
        contents=_contents(prompt, packed),   # + the figures being cited (9.6, gap G7)
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            # ModelDraft, from shared/documind_schemas.py - the same class 3.2 teaches.
            # The model cites by [Source N] and QUOTES the words it relied on; resolve()
            # below turns that into Citations with ids, pages and scores it never saw.
            response_schema=ModelDraft,
            max_output_tokens=max_tokens,
            # NO temperature / top_p / top_k: gemini-3.6-flash ignores all
            # three, and passing them reads like a knob that does something.
            # thinking_level, not thinking_budget=0 - Gemini 3.x thinking
            # cannot be switched off, and budget=0 is a 2.5-era setting.
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
            **_cache_kwargs(tenant_id, model),
        ),
    )


def _exhausted_tier(e: BaseException, model: str) -> bool:
    """A 429 from a ROUTED tier (10.3) is a quota on that tier, not an outage: the default model answers
    instead and the row says so. The first routed eval (10 September) served two rows a 500 because the Pro
    tier's quota ran out (F45). A 429 from the default model itself, or from a tuned endpoint that IS the
    default, is still the caller's error to see."""
    return isinstance(e, errors.APIError) and getattr(e, "code", None) == 429 and model != settings.generator_model


def _call_or_fallback(prompt: str, packed: list[dict], tenant_id: str | None, max_tokens: int, model: str):
    """_call, and on an exhausted tier the same call on the default model. Returns (response, model that answered)."""
    try:
        return _call(prompt, packed, tenant_id, max_tokens, model), model
    except errors.APIError as e:
        if not _exhausted_tier(e, model):
            raise
        log.warning(json.dumps({"event": "tier_exhausted", "tenant": tenant_id, "model": model,
                                "fallback": settings.generator_model}))
        return _call(prompt, packed, tenant_id, max_tokens, settings.generator_model), settings.generator_model


def _quote_limit() -> int:
    """The contract's ceiling on a draft quote (shared/documind_schemas.py: 200 characters)."""
    for m in DraftCitation.model_fields["quote"].metadata:
        if getattr(m, "max_length", None):
            return int(m.max_length)
    return 200


def _problems(e: ValidationError) -> list[str]:
    return [".".join(str(x) for x in err["loc"]) + ": " + err["msg"] for err in e.errors()][:4]


def _draft(r) -> ModelDraft | None:
    """The model's structured answer, or None when there is nothing parseable to answer from.

    The SDK validates the JSON against ModelDraft and hands back None on ANY violation. The
    third live eval (7 Sept 2026) found the violation that matters: a quote longer than the
    contract's 200 characters. A statute provision is one long sentence, "exact words from
    that source" invites the model to copy it whole, and nine correct answers were thrown
    away for an excerpt that was merely long - resolve() would have cut it to 500 anyway,
    but never got the chance. So: the JSON is read, an over-long quote is trimmed to the
    contract, the draft is validated again, and the repair is logged with the field that
    failed. Anything else that fails validation is logged the same way and stays a failure.
    """
    if isinstance(r.parsed, ModelDraft):
        return r.parsed
    if r.parsed:
        return ModelDraft.model_validate(r.parsed)
    try:
        text = r.text
    except Exception:
        text = None
    if not text:
        return None
    try:
        obj = json.loads(text)
    except ValueError:
        return None
    try:
        return ModelDraft.model_validate(obj)
    except ValidationError as e:
        problems = _problems(e)
    limit = _quote_limit()
    for c in obj.get("citations") or []:
        if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > limit:
            c["quote"] = c["quote"][: limit - 3].rstrip() + "..."
    try:
        draft = ModelDraft.model_validate(obj)
    except ValidationError as e:
        log.error(json.dumps({"event": "generation_invalid", "problems": _problems(e)}))
        return None
    log.warning(json.dumps({"event": "generation_repaired", "problems": problems}))
    return draft


def _finish_reason(r) -> str:
    """Why the model stopped: STOP, MAX_TOKENS, SAFETY, ... or why it never started."""
    cands = r.candidates or []
    if not cands:
        block = getattr(getattr(r, "prompt_feedback", None), "block_reason", None)
        return f"BLOCKED:{getattr(block, 'name', block)}" if block else "NO_CANDIDATES"
    fr = getattr(cands[0], "finish_reason", None)
    return getattr(fr, "name", str(fr)) if fr is not None else "UNKNOWN"



def generate(query: str, chunks: list[dict], tenant_id: str | None = None, model: str | None = None,
             backend: str | None = None) -> RAGResponse:
    """`model`: the routed tier (10.3) or the generator model; a tuned endpoint (10.1) is a value of either.
    `backend`: vertex (google.genai) or gateway (11.3's LiteLLM, where `model` names a route) - the setting, or 11.4's
    per-tenant pin."""
    model = model or settings.generator_model
    backend = backend or settings.model_backend
    if backend == "gateway":
        model = _gateway_route(model)
    # 4.5's context_budget, wired. Until now settings.max_context_tokens was
    # declared in config.py and read by nothing, so a long retrieval went to
    # the model whole and the budget was decorative. _pack() packs inside what is
    # left of it after the prompt's fixed parts (_budget, 12 September 2026).
    context, packed = _pack(query, chunks)
    prompt = f"{SYSTEM}{_dated_rule(packed)}\\n\\nContext:\\n{context}\\n\\nQuestion: {query}"

    if backend == "gateway":
        r = _gateway_call(prompt, packed, tenant_id, settings.max_answer_tokens, model)
    else:
        r, model = _call_or_fallback(prompt, packed, tenant_id, settings.max_answer_tokens, model)
    billed = {**_usage(r), "cost_usd": getattr(r, "cost_usd", None)}   # every paid attempt counts: this one, and the retry
    draft = _draft(r)
    if draft is None and _finish_reason(r) == "MAX_TOKENS":
        # Cut off, not refused. A statute answer with its quotes - and, on the 3.x family,
        # the thinking drawn from the same budget before them - can outrun the first cap.
        # Once more with three times the room. The second live eval (7 Sept 2026) scored
        # fourteen of these as refusals: every one a real document with a real answer.
        log.warning(json.dumps({"event": "generation_truncated", "tenant": tenant_id,
                                "max_output_tokens": settings.max_answer_tokens,
                                "tokens_out": r.usage_metadata.candidates_token_count or 0,
                                "thoughts": getattr(r.usage_metadata, "thoughts_token_count", None)}))
        if backend == "gateway":
            r = _gateway_call(prompt, packed, tenant_id, settings.max_answer_tokens * 3, model)
        else:
            r, model = _call_or_fallback(prompt, packed, tenant_id, settings.max_answer_tokens * 3, model)
        # The first attempt was billed too. Rebinding `r` dropped it from the row until 12 September 2026 (6.2):
        # a truncated-then-retried answer cost the tenant two calls and was counted as one.
        billed = _add(billed, {**_usage(r), "cost_usd": getattr(r, "cost_usd", None)})
        draft = _draft(r)
    if draft is None:
        # NOT a refusal. This used to substitute answerable=False, confidence="low" and no
        # citations - the exact shape of the model declining - so a parse failure scored as
        # the lane saying "not in my documents" and nothing anywhere said otherwise. A 502
        # is what happened; the gate counts it as plumbing, which is where it belongs.
        reason = _finish_reason(r)
        log.error(json.dumps({"event": "generation_unparsed", "tenant": tenant_id,
                              "finish_reason": reason,
                              "tokens_out": r.usage_metadata.candidates_token_count or 0}))
        raise HTTPException(502, f"generation produced no parseable answer ({reason})")
    # Resolve against PACKED, not `chunks`. The model numbered what it SAW, and the budget
    # may have dropped something in between - indexing the pre-budget list shifted every
    # citation after a dropped chunk by one, silently. Gap G1 found it; resolve() owns it.
    ans = resolve(draft, packed)
    return RAGResponse(
        **ans.model_dump(),
        model=model,
        backend=backend,
        **billed,            # tokens_in, tokens_out (thinking included), cached_tokens, cost_usd - across every attempt
        latency_ms=0,
    )


def generate_stream(query: str, chunks: list[dict], tenant_id: str | None = None, model: str | None = None,
                    backend: str | None = None):
    """Yield ("packed", chunks) once, then ("token", text) as the model produces it, then ("usage", dict).

    REAL streaming, unlike the version this replaced, which called the blocking
    generate(), waited for the whole answer and then split it on whitespace.
    That looked identical in a browser and had exactly the same time to first
    token as /v1/query - the feature was absent and the demo could not show it.

    No response_schema here on purpose: you cannot usefully stream structured
    JSON. The citations do not need it - they are known BEFORE generation - but
    they are the PACKED set, not the reranked pool: the budget may drop a chunk
    between the reranker and the prompt, and a citation to a passage the model
    never read is not a citation. So the first event is the packed list
    (12 September 2026, R06) and /v1/stream cites from it before the first token.
    """
    context, packed = _pack(query, chunks)
    yield "packed", packed
    prompt = f"{SYSTEM}{_dated_rule(packed)}\\n\\nContext:\\n{context}\\n\\nQuestion: {query}"

    model = model or settings.generator_model
    backend = backend or settings.model_backend
    if backend == "gateway":
        yield from _gateway_stream(prompt, packed, tenant_id, _gateway_route(model))
        return
    yield from _vertex_stream(prompt, packed, tenant_id, model)


def _vertex_stream(prompt: str, packed: list[dict], tenant_id: str | None, model: str):
    """The tokens and the usage from google.genai - and, on an exhausted routed tier before the first token,
    the default model's instead (F45). Below generate_stream so the packed event is yielded exactly once,
    whichever model ends up answering."""
    usage = {"tokens_in": 0, "tokens_out": 0}
    yielded = 0
    try:
        for part in _client_for(model).models.generate_content_stream(
            model=model,
            contents=_contents(prompt, packed),   # + the figures being cited (9.6, gap G7)
            config=types.GenerateContentConfig(
                max_output_tokens=settings.max_answer_tokens,
                thinking_config=types.ThinkingConfig(thinking_level="LOW"),
                **_cache_kwargs(tenant_id, model),
            ),
        ):
            if part.text:
                yielded += 1
                yield "token", part.text
            if part.usage_metadata:
                # The last chunk carries the totals; earlier ones may carry partials. Thinking counts as
                # output (6.2), and 10.2's caching only shows up in the bill if something records it.
                # This is that something.
                usage = _usage(part)
    except errors.APIError as e:
        # An exhausted routed tier before the first token: the default model streams instead (F45).
        if yielded or not _exhausted_tier(e, model):
            raise
        log.warning(json.dumps({"event": "tier_exhausted", "tenant": tenant_id, "model": model,
                                "fallback": settings.generator_model}))
        yield from _vertex_stream(prompt, packed, tenant_id, settings.generator_model)
        return
    # The model that answered, for the usage row: the tier, or the default it fell back to.
    yield "usage", {**usage, "model": model}
'''
with open('generator.py', 'w') as f: f.write(GENERATOR_PY)
print('generator.py written')


In [ ]:
AUTH_PY = '''
"""IAP identity and tenant membership for the DocuMind RAG API.

The tenant is something you ARE, not something you send. The previous version
read x-user-email and x-tenant-id straight off the request, so anyone who could
reach the service could claim any tenant - and Cloud Run services are reachable
from more places than people expect.

Since 12.8 the verifier itself is shared/iap.py - one implementation for four
surfaces - and it has two legs (gap G4, 2026-09-05):

    x-goog-iap-jwt-assertion   the PERSON, forwarded by the surface they signed
                               in to (documind-ui, documind-chat). Accepted for
                               every audience in IAP_AUDIENCE, and only those.
    Authorization: Bearer      the CALLER's own Google ID token, minted for
                               SELF_URL (lesson 7.3): an agent in a notebook,
                               run_eval.py, smoke.py. No person, so no
                               assertion; the account itself must be on the
                               tenant's roster.

The assertion wins when both are present. Nothing else is read.
"""
import os

from fastapi import HTTPException, Request

from shared import iap
from shared.tenancy import is_member

AUTH_MODE = os.environ.get("AUTH_MODE", "iap")        # iap | dev
# This service's own URL - the audience every caller mints an ID token for.
# Unset means the bearer leg is off and only forwarded assertions are accepted.
SELF_URL = os.environ.get("SELF_URL", "")


def verify_iap(request: Request) -> dict:
    """Return {"email": ..., "via": iap|iam|dev} for the caller, or raise 401."""
    if AUTH_MODE == "dev":
        # Local and test only. Set AUTH_MODE=iap everywhere else; run-service.yaml
        # does exactly that, so dev mode cannot reach prod by accident.
        email = request.headers.get("x-user-email")
        if not email:
            raise HTTPException(401, "missing identity headers")
        return {"email": email.lower(), "via": "dev"}
    try:
        return iap.identity(request.headers, bearer_audience=SELF_URL or None)
    except iap.IapError as e:
        # 401, not 403: we do not know who this is. This also fails closed on a
        # missing IAP_AUDIENCE - verifying without an audience accepts a token
        # minted for ANY service.
        raise HTTPException(401, str(e))


def enforce_membership(email: str, tenant_id: str) -> None:
    """403 on mismatch - a real authorisation check, not a header comparison.

    One roster, read one way: shared/tenancy.py, the same document the frontend
    and the chat service consult, so a membership change lands everywhere at once.
    """
    if not is_member(email, tenant_id):
        raise HTTPException(403, "not a member of this tenant")
'''

with open('auth.py', 'w') as f: f.write(AUTH_PY)
print('wrote auth.py')


In [ ]:
MAIN_PY = '''
import time, json, logging, os, sys
from contextlib import contextmanager
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from opentelemetry import trace
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter
from schemas import QueryRequest, RAGResponse, RAGAnswer, FILTER_KEYS
from retriever import retrieve, rerank, rerank_fell_back, _fs, embed_query
from generator import generate, generate_stream, _client as _gen_client
from config import settings, RETRIEVAL_BACKENDS, MANAGED_BACKENDS
from auth import verify_iap, enforce_membership
from shared.tenancy import policy_of              # the tenant's data_region, one normalisation (13 September 2026, evening)
from cost import price
from router import classify
from breakers import choose_model
from budget import record, spend_pct
import semantic_cache                     # 12.6's answer cache, behind SEMANTIC_CACHE=on (the RAG plan, W4)

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)  # bare JSON -> Cloud Run jsonPayload
log = logging.getLogger("documind-api")

trace.set_tracer_provider(TracerProvider())
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(CloudTraceSpanExporter(project_id=settings.project_id)))
tracer = trace.get_tracer(__name__)


def _ms(since: float) -> int:
    return int((time.perf_counter() - since) * 1000)


@contextmanager
def stage(stages: dict, name: str):
    """One stage of an answer on its own clock AND its own span. A p95 that moved says nothing about
    which stage moved it, so the row carries retrieve_ms / rerank_ms / generate_ms beside latency_ms
    (tenant_daily reads them, 12.3; make usage groups them), and the same names are spans in Cloud
    Trace for the one slow request someone is looking at."""
    t = time.perf_counter()
    with tracer.start_as_current_span(name):
        try:
            yield
        finally:
            stages[f"{name}_ms"] = _ms(t)


app = FastAPI(title="DocuMind API", version="1.0.0")
FastAPIInstrumentor.instrument_app(app)
# 12.6: gen_ai spans - what ran, not what was said. telemetry.py instruments the google-genai SDK with content capture
# off (NO_CONTENT), so a trace carries the model, the tokens, the latency and the finish reason and never a prompt.
# The import is guarded: an instrumentation that cannot load is a missing span, never a missing API.
try:
    import telemetry  # noqa: E402,F401
except Exception as _e:  # noqa: BLE001
    log.warning(json.dumps({"event": "telemetry_not_instrumented", "error": type(_e).__name__}))
# 9.4's Media Studio, adopted (gap G8): /v1/media/generate and /v1/media/upload-url, behind the
# same verify_iap and the same roster check, writing the same usage row shape with modality=image.
from media import router as media_router  # noqa: E402
app.include_router(media_router)

app.add_middleware(CORSMiddleware,
    allow_origins=["https://documind.example.com"],
    allow_methods=["POST","GET"], allow_headers=["*"])

# verify_iap and enforce_membership live in auth.py: identity is verified
# against the IAP assertion, and the tenant is checked against the Firestore
# roster rather than compared to a header the caller sent.

def modality_of(kinds) -> str:
    """What the answer was made from (gap G8). A single video segment makes the row
    'video'; a figure or a table makes it 'image'; otherwise text. tenant_daily groups
    by it, so media spend per tenant is a query, not a guess."""
    kinds = set(kinds)
    if "segment" in kinds:
        return "video"
    if kinds & {"figure", "table"}:
        return "image"
    return "text"


def choose_model_for(query: str) -> str:
    """10.3, wired behind a flag. ROUTING=on: the classifier (router.py, flash-lite, one short call) names
    the question's tier and the budget breaker (breakers.py) picks the model for it - Pro for a complex
    question while the month is under 80% of budget, flash-lite for everything simple, and nothing
    dearer than flash once the month is past it. Off: the generator model serves everything, which is
    what the lane ran until Module 10. A classifier failure is never an outage: it falls back."""
    if settings.routing != "on":
        return settings.generator_model
    try:
        tier = classify(query, _gen_client).value
        return choose_model(tier, spend_pct(_fs(), settings.budget_usd, settings.spend_pct_override))
    except Exception as e:  # noqa: BLE001
        log.warning(json.dumps({"event": "routing_fallback", "error": type(e).__name__}))
        return settings.generator_model


_TENANT_SETTINGS: dict = {}


def tenant_settings(tenant_id: str) -> dict:
    """11.4's "pin one tenant": tenant_settings/{tenant} may name a model_backend and a generator_model, and the
    residency customer's answers come from the self-hosted route while everyone else's come from Gemini. Since
    13 September 2026 (evening) the same document may pin a retrieval_backend and declares data_region - where the
    tenant's text may be held (shared/tenancy.py; `in` unless it says `any`). Read once a minute per tenant; a
    missing document or a failed read is the global setting (and the strict policy). A field edit, never a redeploy."""
    now = time.time()
    hit = _TENANT_SETTINGS.get(tenant_id)
    if hit and now - hit[0] < 60:
        return hit[1]
    try:
        snap = _fs().collection("tenant_settings").document(tenant_id).get()
        doc = (snap.to_dict() or {}) if snap.exists else {}
    except Exception:  # noqa: BLE001 - the pin is a convenience; the setting is the default
        doc = {}
    _TENANT_SETTINGS[tenant_id] = (now, doc)
    return doc


def choose_for(req) -> tuple[str, str, str]:
    """(model backend, model, retrieval backend) for this request: the tenant's pins first, else the routed tier (10.3)
    and the settings. A pinned retrieval_backend the deployment cannot serve - an unknown name, or a managed store
    under RETRIEVAL_MODE=hybrid (the pair config.py refuses at startup) - is ignored with a line, never a 500."""
    ts = tenant_settings(req.tenant_id)
    backend = ts.get("model_backend") or settings.model_backend
    model = ts["generator_model"] if ts.get("generator_model") else choose_model_for(req.query)
    pin = ts.get("retrieval_backend")
    retrieval = settings.retrieval_backend
    if pin and pin != retrieval:
        if pin in RETRIEVAL_BACKENDS and not (pin in MANAGED_BACKENDS and settings.retrieval_mode == "hybrid"):
            retrieval = pin
        else:
            log.warning(json.dumps({"event": "retrieval_pin_ignored", "tenant": req.tenant_id, "retrieval_backend": pin,
                                    "served": retrieval, "why": f"one of {'|'.join(RETRIEVAL_BACKENDS)}, and a managed store cannot fuse hybrid"}))
    return backend, model, retrieval


def retrieval_backend_for(tenant_id: str, backend: str) -> tuple[str, int]:
    """The backend that will serve, held against the tenant's data_region (13 September 2026, evening): a managed
    store (config.MANAGED_BACKENDS) for a tenant whose text may not leave India - `in`, which an absent policy also
    means - is the kit's own index instead, with policy_fallback=1 on the row: the deployment's own backend when
    that is one (vector, firestore), else Firestore, which holds every embedding the worker wrote. Never the store,
    never a 500; the row says it happened, so a day of fallbacks is a count."""
    ts = tenant_settings(tenant_id)
    if backend in MANAGED_BACKENDS and policy_of(ts) != "any":
        own = settings.retrieval_backend if settings.retrieval_backend not in MANAGED_BACKENDS else "firestore"
        return own, 1
    return backend, 0


def _fingerprint(tenant_id: str) -> str:
    """The ledger's corpus fingerprint (12.5: ledger/{tenant}; the identity 10.2's context cache follows too), or ""
    on a lane that has never reindexed. The answer cache is keyed on it: a reindex makes every earlier answer a miss."""
    snap = _fs().collection("ledger").document(tenant_id).get()
    return ((snap.to_dict() or {}).get("fingerprint") or "") if snap.exists else ""


def _semantic_hit(req, qvec, fingerprint):
    """12.6's answer cache, when SEMANTIC_CACHE=on: the stored answer for a near-enough earlier question of this tenant
    under this corpus, as a RAGResponse that cost nothing and cites what the original cited - or None. A cache that
    fails is a miss and a warning, never an error: the answer is still one retrieval away."""
    if settings.semantic_cache != "on":
        return None
    try:
        hit = semantic_cache.lookup(_fs(), req.tenant_id, qvec, fingerprint,
                                    scope=semantic_cache.scope_of(req.filters, req.top_k, settings.prompt_version),
                                    question=req.query)   # the exact rung first: the same words never reach the vector search
    except Exception as e:  # noqa: BLE001
        log.warning(json.dumps({"event": "semantic_cache_failed", "tenant": req.tenant_id, "error": type(e).__name__}))
        return None
    if not hit:
        return None
    return RAGResponse(**hit["answer"], model=hit.get("model") or settings.generator_model, backend="cache",
                       cost_usd=0.0, tokens_in=0, tokens_out=0, latency_ms=0, cache_hit="semantic")


def _semantic_store(req, qvec, ans, fingerprint) -> None:
    """After /v1/query has an answer the caller is getting - answerable, cited, not blocked: a refusal is not worth a
    day and a blocked answer is not worth anything. Only the contract is stored (RAGAnswer: answer, citations,
    confidence, answerable), never the envelope. Failing to store is a warning, never a failed request."""
    if settings.semantic_cache != "on" or not (ans.answerable and ans.citations):
        return
    try:
        semantic_cache.store(_fs(), req.tenant_id, req.query, qvec,
                             RAGAnswer.model_validate(ans.model_dump()).model_dump(), fingerprint, model=ans.model,
                             scope=semantic_cache.scope_of(req.filters, req.top_k, settings.prompt_version))
    except Exception as e:  # noqa: BLE001
        log.warning(json.dumps({"event": "semantic_cache_store_failed", "tenant": req.tenant_id, "error": type(e).__name__}))


def check_filters(filters: dict | None) -> None:
    """The request's filters, or a 400 (12 September 2026, R06). A key that is not a restrict namespace on the
    index and a field on the row filters nothing on one path and everything on another; tenant_id and `current`
    are the roster's and the ledger's, never the caller's - a body field is a header in disguise. A typo is a 400
    that names the allowed keys, not an empty pool that reads like an honest "nothing found"."""
    if not filters:
        return
    bad = sorted(set(filters) - set(FILTER_KEYS))
    if bad:
        raise HTTPException(400, f"unknown filter key(s) {', '.join(bad)}; allowed: {', '.join(FILTER_KEYS)}")
    for k, v in filters.items():
        if not isinstance(v, str) or not v:
            raise HTTPException(400, f"filter {k} must be a non-empty string")


EMPTY_POOL_ANSWER = ("The corpus holds nothing near this question: no passage of this tenant's current documents was "
                     "retrieved, so there is nothing to answer from and nothing to cite. Check that the document you "
                     "expect is ingested and current, or ask with the words it uses.")


def empty_pool_answer(model: str, stages: dict) -> RAGResponse:
    """The answer for an empty pool (12 September 2026, R06): retrieval found no candidate, so there is nothing for
    the reranker to order and nothing for the model to read - a model call would buy an invented answer or a refusal
    at full price. A refusal in the contract's own shape (answerable=False, no citations, confidence low), backend
    "none", zero tokens, zero cost; the rerank and generate clocks read 0 because they never ran. Never stored in the
    answer cache (_semantic_store refuses an unanswerable answer), and the row's answerable=False is what the
    unanswerable alert counts: a question the corpus cannot reach is its business."""
    stages["rerank_ms"] = stages["generate_ms"] = 0
    return RAGResponse(answer=EMPTY_POOL_ANSWER, citations=[], confidence="low", answerable=False, model=model,
                       backend="none", cost_usd=0.0, tokens_in=0, tokens_out=0, latency_ms=0, cache_hit="none")


def usage_row(req, user, ans_tokens_in, ans_tokens_out, cached, latency_ms,
              answerable, confidence, surface, modality="text", model=None, backend=None, cost_usd=None, guard="off",
              stages=None, retrieval_backend=None):
    """The ONE shape every observability consumer reads.

    tenant_daily.sql selects exactly these fields, so a column added there
    without a field added here is a column of NULLs that looks like it works.
    `model` is the model that ANSWERED - the routed tier or the tuned endpoint, when there is one -
    priced at its own rate (cost.py), so tenant_daily's cost column is what was billed.
    """
    model = model or settings.generator_model
    stages = stages or {}
    # The gateway prices what it served, fallbacks included (Module 11); otherwise cost.py's rate for the model.
    cost = cost_usd if cost_usd is not None else price(model, ans_tokens_in, ans_tokens_out, cached)["usd"]
    return {"event": surface, "tenant": req.tenant_id, "user": user["email"],
            "tokens_in": ans_tokens_in, "tokens_out": ans_tokens_out,
            "cached_tokens": cached, "cost_usd": round(cost, 6),
            "latency_ms": latency_ms, "answerable": answerable,
            # Where the time went: the three stages on their own clocks (stage() above) and the size of the pool
            # the reranker saw - the number TOP_K_RETRIEVE sets and evals/ablate.py decides. "p95 is 3 s" is a
            # page; "generate is 2.6 s of it" is a fix. A row from before these fields is NULL in the view, not 0.
            "retrieve_ms": stages.get("retrieve_ms", 0), "rerank_ms": stages.get("rerank_ms", 0),
            "generate_ms": stages.get("generate_ms", 0), "pool": stages.get("pool", 0),
            # 1 when the Ranking API did not answer and the pool stood in by retrieval score (retriever.rerank):
            # a degraded order, counted - the rerank_fallback log event beside it carries the error type.
            "rerank_fallback": stages.get("rerank_fallback", 0),
            "confidence": confidence,
            # An explicit 0/1 beside the boolean. Cloud Logging's
            # value_extractor pulls a NUMBER out of a log entry; it cannot
            # turn true/false into one, so the metric behind the
            # unanswerable-rate alert would have nothing to read.
            "unanswerable_flag": 0 if answerable else 1,
            "model_backend": backend or settings.model_backend,    # what ANSWERED: vertex, or the gateway (Module 11)
            "guard": guard,                                       # 12.6: off | pass | blocked_response - what Model Armor said, when asked
            "model": model,
            "prompt_version": settings.prompt_version,
            "retrieval_mode": settings.retrieval_mode,
            # 4.6's graph on the lane (13 September 2026): the switch, and how many of the pool's chunks the walk put there
            "retrieval_graph": settings.retrieval_graph, "graph_chunks": stages.get("graph_chunks", 0),
            # P9.4: which store served the pool - the EFFECTIVE one (the tenant's pin, or the kit's index after the
            # policy fallback; the setting when the handler passed none), how many of its chunks a managed store put
            # there (the rest fell back), and 1 when the tenant's data_region sent a managed backend to the kit's index
            "retrieval_backend": retrieval_backend or settings.retrieval_backend, "managed_chunks": stages.get("managed_chunks", 0),
            "policy_fallback": stages.get("policy_fallback", 0),
            "modality": modality, "surface": surface,
            # 8.7's question - which harness costs what - answered from the warehouse: the
            # chat service labels its brain on every call, the UI's own stream is "ui".
            "brain": getattr(req, "brain", None) or "ui"}


@app.get("/health")
def health(): return {"status": "ok"}

@app.get("/version")
def version():
    # What is actually serving. When an answer changes and no one deployed,
    # this is the first thing to check - and it is the same triple that goes
    # on every log line and every span.
    return {"model_backend": settings.model_backend,
            "generator_model": settings.generator_model,
            "prompt": f"{settings.prompt_id}@{settings.prompt_version}",
            "retrieval_mode": settings.retrieval_mode,
            "retrieval_backend": settings.retrieval_backend,   # vector | firestore | rag_engine | vertex_search: the DEFAULT; a tenant's pin and its data_region decide per request (the row's retrieval_backend is the effective one)
            "retrieval_graph": settings.retrieval_graph,   # 4.6's graph: off | on | auto (13 September 2026)
            "graph_backend": settings.graph_backend,       # firestore | spanner (16 September 2026): spanner seeds the walk by meaning
            # 12 September 2026: the embedding the query vector comes from - the same pair the worker stamps on
            # every row - and whether the ledger's pre-filter is on. A reindex that "changed nothing" and a
            # retrieval that "got worse" both start here.
            "embedding": f"{settings.embed_model}@{settings.embedding_version}",
            "retrieval_current_only": settings.retrieval_current_only,
            "semantic_cache": settings.semantic_cache,   # 12.6's answer cache: off | on (smoke.py asks twice when on)
            "git_sha": os.environ.get("GIT_SHA", "unknown")}

@app.get("/v1/sources")
def sources(tenant_id: str, user=Depends(verify_iap)):
    """The versions view (12 September 2026): the tenant's ledger - every source's current version, its object
    generation, what the last reindex cost (chunks reused by hash, embedded, retired), the date it declares, when
    it landed - and the corpus fingerprint the cache is keyed to. Read-only, and only for a tenant the caller is
    on the roster of: one customer's ledger is not another's to read. The UI's Documents page renders it;
    `make sources TENANT=` prints the same rows from the shell (reconcile.py --report). `mirrored` (13 September 2026,
    evening) is where the version is HELD: the managed stores that confirmed it, with regions; empty is the kit's rows
    only - beside the tenant's data_region, the policy those copies were judged under."""
    enforce_membership(user["email"], tenant_id)
    fs = _fs()
    rows = []
    for s in fs.collection("sources").where("tenant_id", "==", tenant_id).stream():
        d = s.to_dict() or {}
        at = d.get("indexed_at")
        rows.append({"name": d.get("name"), "status": d.get("status"), "doc_key": d.get("doc_key"),
                     "generation": d.get("generation"), "chunks": d.get("chunks"),
                     "reused": d.get("reused"), "embedded": d.get("embedded"), "retired": d.get("retired"),
                     "effective_from": d.get("effective_from"),
                     "embedding": f"{d.get('embedding_model') or '?'}@{d.get('embedding_version') or '?'}",
                     "indexed_at": at.isoformat() if hasattr(at, "isoformat") else None,
                     "mirrored": d.get("mirrored") or {}})
    rows.sort(key=lambda r: r["name"] or "")
    led = fs.collection("ledger").document(tenant_id).get()
    l = (led.to_dict() or {}) if led.exists else {}
    return {"tenant_id": tenant_id, "fingerprint": l.get("fingerprint"), "versions": l.get("versions"),
            "last_event": l.get("last_event"), "data_region": policy_of(tenant_settings(tenant_id)), "sources": rows}

@app.get("/ready")
def ready():
    # Lazy-init resource probes keep cold start fast; only warm when ready is probed
    from config import settings
    from retriever import _genai_client, _index_endpoint, _fs, _rag, _search
    _ = _genai_client(); _ = _fs()
    if settings.retrieval_backend == "vector":      # a Firestore-only deployment has no endpoint to warm
        _ = _index_endpoint()
    elif settings.retrieval_backend == "rag_engine":   # P9.4: vertexai's init, before the first question pays for it
        _ = _rag()
    elif settings.retrieval_backend == "vertex_search":   # R4: the search client, likewise
        _ = _search()
    return {"status": "ready"}

def _guard():
    """12.6's guard, imported only when the revision says ARMOR=on: guard.py builds its Model Armor client and names
    its template at import, and a revision that never asked for the guard must neither pay for the client nor fail on
    a template it does not have."""
    import guard  # noqa: WPS433
    return guard


def screen_prompt(text: str, tenant_id: str) -> str:
    """Before retrieval, never after: an injection that reaches the retriever has already chosen which documents the
    model reads. A block is a 400 with the reason - a refusal dressed as an answer would hide the rate."""
    if settings.armor != "on":
        return "off"
    ok, reason = _guard().check_prompt(text)
    if not ok:
        log.info(json.dumps({"event": "guard", "tenant": tenant_id, "verdict": "blocked_prompt", "reason": reason}))
        raise HTTPException(400, reason)
    return "pass"


def screen_response(text: str, guard: str) -> tuple[str, str]:
    """On the buffered final answer - a token stream cannot be screened, so /v1/stream holds its tokens when the guard
    is on. Returns the row's verdict and, when blocked, the reason the caller raises AFTER the row is logged."""
    if settings.armor != "on":
        return guard, ""
    ok, reason = _guard().check_response(text)
    return ("pass", "") if ok else ("blocked_response", reason)


@app.post("/v1/query", response_model=RAGResponse)
def query(req: QueryRequest, user=Depends(verify_iap)):
    enforce_membership(user["email"], req.tenant_id)
    check_filters(req.filters)                            # a 400 before any work: an unknown key is a typo, not an empty pool
    t0 = time.time()
    backend, model, rbackend = choose_for(req)
    guard = screen_prompt(req.query, req.tenant_id)       # 12.6: before retrieval, or not at all (ARMOR=off)
    stages: dict = {}
    rbackend, stages["policy_fallback"] = retrieval_backend_for(req.tenant_id, rbackend)   # the tenant's data_region, per request
    stages["retrieval_backend"] = rbackend               # the backend chosen for THIS request (16 September 2026): the row carried it, the answer did not
    fingerprint = _fingerprint(req.tenant_id) if settings.semantic_cache == "on" else ""
    with stage(stages, "retrieve"):
        qvec = embed_query(req.query)                     # once: the answer cache and the retrieval share it
        hit = _semantic_hit(req, qvec, fingerprint)
        chunks = [] if hit else retrieve(req.query, req.tenant_id, req.top_k, req.filters, vec=qvec, backend=rbackend)
    stages["pool"] = len(chunks)                          # what the reranker sees: TOP_K_RETRIEVE, as served
    stages["graph_chunks"] = sum(1 for c in chunks if c.get("found_by") == "graph")   # 4.6's walk, counted
    stages["managed_chunks"] = sum(1 for c in chunks if c.get("found_by") in MANAGED_BACKENDS)   # P9.4 / R4: the store's share of the pool
    stages["vector_chunks"] = sum(1 for c in chunks if c.get("found_by") == "vector")   # the ANN tier's share (16 September 2026): 0 while retrieval_backend is `vector` means the Firestore rung answered
    if hit:
        ans = hit                                         # served from answer_cache: no reranker, no model
        stages["rerank_ms"] = stages["generate_ms"] = 0
    elif not chunks:
        ans = empty_pool_answer(model, stages)            # nothing retrieved: no reranker, no model, nothing stored
    else:
        with stage(stages, "rerank"):
            chunks = rerank(req.query, chunks, req.top_k, tenant_id=req.tenant_id)
        if rerank_fell_back(chunks):
            stages["rerank_fallback"] = 1                 # the pool by retrieval score stood in for the Ranking API
        with stage(stages, "generate"):
            ans = generate(req.query, chunks, req.tenant_id, model=model, backend=backend)
    ans.latency_ms = int((time.time() - t0) * 1000)
    ans.stages = dict(stages)                             # the same clocks in the answer, for the caller
    guard, reason = screen_response(ans.answer, guard)   # 12.6: the buffered final answer, never a token
    if not (hit or reason):
        _semantic_store(req, qvec, ans, fingerprint)      # only what the caller is getting: answerable, cited, not blocked
    # ans.model is the model that ANSWERED: the routed tier, the default it fell back to on a 429 (F45), or the
    # gateway route (Module 11); ans.backend says which door it went through.
    row = usage_row(req, user, ans.tokens_in, ans.tokens_out, getattr(ans, "cached_tokens", 0),
                    ans.latency_ms, ans.answerable, ans.confidence, "query",
                    modality=modality_of(c.kind for c in ans.citations), model=ans.model or model,
                    backend=ans.backend, cost_usd=ans.cost_usd, guard=guard, stages=stages, retrieval_backend=rbackend)
    log.info(json.dumps(row))
    _record(row["cost_usd"])
    if reason:
        raise HTTPException(502, reason)                  # the row above says blocked_response; the caller gets the reason
    return ans


def _record(usd: float) -> None:
    """The month's counter (budget.py). Never in the answer's way: a counter that fails fails quietly."""
    try:
        record(_fs(), usd)
    except Exception as e:  # noqa: BLE001
        log.warning(json.dumps({"event": "budget_record_failed", "error": type(e).__name__}))

@app.post("/v1/stream")
def stream(req: QueryRequest, user=Depends(verify_iap)):
    enforce_membership(user["email"], req.tenant_id)
    check_filters(req.filters)                            # a 400 before the stream starts, like the guard's
    guard = screen_prompt(req.query, req.tenant_id)       # 12.6: before the stream starts, so a block is a 400, not a broken stream
    def sse():
        t0 = time.time()
        backend, model, rbackend = choose_for(req)
        # The same three clocks as /v1/query, without the spans: a generator suspended between tokens is
        # no place to hold a span's context, and the trace already carries the request's own span.
        stages: dict = {}
        rbackend, stages["policy_fallback"] = retrieval_backend_for(req.tenant_id, rbackend)
        stages["retrieval_backend"] = rbackend
        fingerprint = _fingerprint(req.tenant_id) if settings.semantic_cache == "on" else ""
        tick = time.perf_counter()
        qvec = embed_query(req.query)
        hit = _semantic_hit(req, qvec, fingerprint)      # the stream reads the answer cache; only /v1/query fills it
        chunks = [] if hit else retrieve(req.query, req.tenant_id, req.top_k, req.filters, vec=qvec, backend=rbackend)
        stages["retrieve_ms"], stages["pool"] = _ms(tick), len(chunks)
        stages["graph_chunks"] = sum(1 for c in chunks if c.get("found_by") == "graph")
        stages["managed_chunks"] = sum(1 for c in chunks if c.get("found_by") in MANAGED_BACKENDS)
        stages["vector_chunks"] = sum(1 for c in chunks if c.get("found_by") == "vector")
        tick = time.perf_counter()
        if chunks:                                   # a hit brought none; an empty pool has nothing to rank
            chunks = rerank(req.query, chunks, req.top_k, tenant_id=req.tenant_id)
            if rerank_fell_back(chunks):
                stages["rerank_fallback"] = 1
        stages["rerank_ms"] = _ms(tick)
        # The empty pool (12 September 2026): nothing retrieved is nothing to read and nothing to cite - one
        # token that says so, then done, with no model call; the row's answerable=False feeds the alert.
        empty = None if (hit or chunks) else empty_pool_answer(model, stages)
        kinds = [c.kind for c in hit.citations] if hit else []      # what the answer was made from, for the row
        for i, c in enumerate(hit.citations if hit else [], 1):
            # A hit's citations are the contract's, resolved when the answer was first given: the same event shape.
            yield f"event: citation\\ndata: {json.dumps({'n': i, 'chunk_id': c.chunk_id, 'source': c.source_uri, 'page': c.page, 'quote': c.quote[:240], 'kind': c.kind, 'media_url': c.media_url, 'start': c.start, 'end': c.end})}\\n\\n"
        usage = {"tokens_in": 0, "tokens_out": 0}
        held = []                                    # 12.6: with the guard on, the answer is screened whole, then sent
        tick = time.perf_counter()
        if hit:                                      # the stored answer as one token: the UI cannot tell, the row can
            usage = {"tokens_in": 0, "tokens_out": 0, "cached_tokens": 0, "cost_usd": 0.0, "model": hit.model, "backend": "cache"}
        elif empty:                                  # the refusal as one token: no model, no cost, backend "none"
            usage = {"tokens_in": 0, "tokens_out": 0, "cached_tokens": 0, "cost_usd": 0.0, "model": empty.model, "backend": "none"}
        for kind, payload in ([("token", hit.answer)] if hit else [("token", empty.answer)] if empty else
                              generate_stream(req.query, chunks, req.tenant_id, model=model, backend=backend)):
            if kind == "packed":
                # Citations first, and from the PACKED set (12 September 2026, R06): they are known before a
                # single token exists, so the UI renders the sources while the answer is written - but the
                # budget may drop a chunk between the reranker and the prompt, and a passage the model never
                # read is not a source. generate_stream yields this list before its first token.
                kinds = [c.get("kind", "text") for c in payload]
                for i, c in enumerate(payload, 1):
                    # The same fields a Citation carries, so the UI renders a figure or a video
                    # segment from the stream exactly as it would from /v1/query (gap G7).
                    yield f"event: citation\\ndata: {json.dumps({'n': i, 'chunk_id': c.get('id'), 'source': c['source_uri'], 'page': c.get('page_start'), 'quote': c['text'][:240], 'kind': c.get('kind', 'text'), 'media_url': c.get('media_url'), 'start': c.get('start'), 'end': c.get('end'), 'effective_from': c.get('effective_from')})}\\n\\n"
            elif kind == "token":
                if settings.armor == "on":
                    held.append(payload)
                else:
                    yield f"event: token\\ndata: {json.dumps({'t': payload})}\\n\\n"
            else:
                usage = payload
        stages["generate_ms"] = 0 if empty else _ms(tick)   # first token to last: the model's whole turn, as the client felt it
        verdict, reason = screen_response("".join(held), guard) if held else (guard, "")
        if reason:
            yield f"event: error\\ndata: {json.dumps({'error': reason})}\\n\\n"
        else:
            for t in held:
                yield f"event: token\\ndata: {json.dumps({'t': t})}\\n\\n"
        model = usage.get("model") or model          # the model that answered (F45: a tier can fall back)
        backend = usage.get("backend") or backend    # and the door it went through (Module 11)
        done = {**usage, "latency_ms": int((time.time() - t0) * 1000), "stages": stages,
                "cache_hit": "semantic" if hit else "none",
                "model": model, "backend": backend,
                "prompt": f"{settings.prompt_id}@{settings.prompt_version}"}
        # The row's verdict: a refusal for the empty pool; otherwise the stream still says answerable (6.3 of the
        # plan gives the stream a real verdict) - the empty pool no longer hides in that.
        answerable, confidence = (False, "low") if empty else (True, hit.confidence if hit else "medium")
        row = usage_row(req, user, usage.get("tokens_in", 0), usage.get("tokens_out", 0),
                        usage.get("cached_tokens", 0), done["latency_ms"],
                        answerable, confidence, "stream",
                        modality=modality_of(kinds), model=model,
                        backend=backend, cost_usd=usage.get("cost_usd"), guard=verdict, stages=stages, retrieval_backend=rbackend)
        log.info(json.dumps(row))
        _record(row["cost_usd"])
        yield f"event: done\\ndata: {json.dumps(done)}\\n\\n"
    return StreamingResponse(sse(), media_type="text/event-stream")
'''
with open('main.py', 'w') as f: f.write(MAIN_PY)
print('main.py written')


In [ ]:
MEDIA_PY = r'''
"""Media Studio endpoints. Lesson 9.4's server half, adopted (gap G8); the upload door moved to the
uploads bucket when Module 9 joined the lane (9 September 2026).

Generation is global; storage is asia-south1. Two things this file does that the notebook
version did not: it checks the tenant ROSTER before spending on a tenant's behalf (the same
enforce_membership every query passes), and it writes a USAGE row - event=media,
modality=image, cost_usd - in the shape tenant_daily reads, so media spend per tenant is a
column and not an estimate. The audit row (who generated what, prompt hashed, never stored)
is written through the one shared audit writer, whose registry names media.generate.

The signed upload URL points at the UPLOADS bucket, under the tenant's prefix. It used to point
at the media bucket, which has no object.finalized notification (eventarc.tf watches uploads
only), so a video uploaded the way 9.4 teaches was stored, billed for thirty days and never
indexed. Under `{tenant}/` in the uploads bucket the same PUT is an ingest: the worker keys its
media branch on the content type the notification carries (12.5, MEDIA_TYPES), describes the
image or the video with Gemini, and the caption or the segments join the same `chunks`
collection every retrieve() reads. Media is a document.

Signing on Cloud Run: the service's credential is a token, so generate_signed_url is handed the
service's email and access token and signs through the IAM signBlob API (_signing_kwargs below),
which needs iam.serviceAccountTokenCreator on ITSELF (sa.tf: api_self_impersonate, the grant the
UI's account already had for its citation URLs). And the URL is authorised AS its signer, so the
API's account also holds objectCreator on the uploads bucket (storage.tf: api_uploads).
"""
import hashlib
import json
import logging
import os
import time
from datetime import timedelta

import google.auth
import google.auth.credentials
import google.auth.transport.requests
from fastapi import APIRouter, Depends, HTTPException
from google import genai
from google.genai import types
from google.cloud import storage
from pydantic import BaseModel, Field

from shared import audit_log
from auth import enforce_membership, verify_iap

router = APIRouter(prefix="/v1/media", tags=["media"])
log = logging.getLogger("documind-api")   # the logger main.py writes usage rows on

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
BUCKET = os.environ.get("MEDIA_BUCKET", f"{PROJECT}-media")            # storage.tf's media bucket: generated assets
UPLOAD_BUCKET = os.environ.get("UPLOAD_BUCKET", f"{PROJECT}-uploads")  # storage.tf's uploads bucket: the one with a notification
DEMO_MODE = os.getenv("DEMO_MODE") == "1"
# The audit row is the record (the media bucket is a cache), and shared/audit_log.py refuses to drop an
# event when AUDIT_BUCKET is unset. The first live generate (9 September 2026) found the API had never
# been given the bucket - only the worker and the admin had - and raised AFTER storing the image: a
# spend with no record. So the route checks before it spends, and 12.2's DEPLOY names the bucket.
AUDIT_BUCKET = os.environ.get("AUDIT_BUCKET", "")
IMAGE_MODEL = "gemini-3.1-flash-image"
IMAGE_USD = 0.039        # per image, verified 2026-09-04 (9.4) - re-verify on the pricing page
# What the ingest worker can turn into chunks (12.5: MEDIA_TYPES plus the document parser). A type
# outside this list would be stored and then fail in the worker, five deliveries later, in the DLQ.
UPLOAD_TYPES = {"image/png", "image/jpeg", "video/mp4", "audio/mpeg", "application/pdf", "text/markdown", "text/plain"}

_client = genai.Client(enterprise=True, project=PROJECT, location="global")
_gcs = storage.Client()

# SIGNING ON CLOUD RUN. The service's credential is a token from the metadata server, not a key, and
# generate_signed_url will not sign with it: "you need a private key to sign credentials" (the first
# live upload URL, 9 September 2026), with iam.serviceAccountTokenCreator granted and unused. Handed
# the service's email and access token instead, the library signs through the IAM signBlob API - the
# path that grant exists for. A credential that can sign itself (a key file, an impersonated
# credential in a notebook) needs nothing. services/frontend/citations.py carries the same twelve lines: the frontend image
# has no shared/ to import them from, and the gate holds the two copies to one shape.
def _signing_kwargs() -> dict:
    creds, _ = google.auth.default()
    if isinstance(creds, google.auth.credentials.Signing):
        return {}
    creds.refresh(google.auth.transport.requests.Request())
    return {"service_account_email": creds.service_account_email, "access_token": creds.token}


class GenerateIn(BaseModel):
    prompt: str = Field(min_length=1, max_length=2000)
    tenant_id: str = Field(min_length=1)


def _usage(tenant_id: str, email: str, cost_usd: float, cached: bool, latency_ms: int) -> dict:
    """The media row. Same field names as usage_row() in main.py where they overlap, so
    tenant_daily's SUM(cost_usd) and GROUP BY modality need no special case."""
    return {"event": "media", "surface": "media", "tenant": tenant_id, "user": email,
            "modality": "image", "model": IMAGE_MODEL, "tokens_in": 0, "tokens_out": 0,
            "cached_tokens": 0, "cost_usd": 0.0 if cached else cost_usd, "cached": cached,
            "latency_ms": latency_ms, "answerable": True, "unanswerable_flag": 0,
            # The stage clocks the API row carries (main.py usage_row): an image has no retrieval and no pool, and
            # its whole latency is the generate stage - so tenant_daily's p95 per stage holds for modality=image too.
            "retrieve_ms": 0, "rerank_ms": 0, "generate_ms": latency_ms, "pool": 0, "rerank_fallback": 0,
            "confidence": "high", "model_backend": "vertex", "prompt_version": "media-v1",
            # no retrieval either (13 September 2026): no store served, no managed share, no policy fallback
            "retrieval_mode": "none", "retrieval_backend": "none", "managed_chunks": 0, "policy_fallback": 0, "brain": "ui"}


@router.post("/generate")
def generate(body: GenerateIn, user=Depends(verify_iap)):
    enforce_membership(user["email"], body.tenant_id)
    if not AUDIT_BUCKET:
        raise HTTPException(503, "AUDIT_BUCKET is not set on this service: refusing to generate what cannot be recorded")
    t0 = time.time()
    key = hashlib.sha256(body.prompt.encode()).hexdigest()[:32]
    blob = _gcs.bucket(BUCKET).blob(f"{body.tenant_id}/gen/{key}.png")

    # DEMO_MODE serves the cached asset instead of paying for it again. A live
    # demo re-running the same prompt eight times is eight bills and eight
    # chances for the network to embarrass you.
    if DEMO_MODE and blob.exists():
        log.info(json.dumps(_usage(body.tenant_id, user["email"], IMAGE_USD, True,
                                   int((time.time() - t0) * 1000))))
        return {"blob": blob.name, "bucket": BUCKET, "cached": True}

    resp = _client.models.generate_content(
        model=IMAGE_MODEL,
        contents=body.prompt,
        config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]))

    png = next((p.inline_data.data for p in resp.candidates[0].content.parts
                if p.inline_data), None)
    if png is None:
        raise HTTPException(502, "model returned no image part")
    blob.upload_from_string(png, content_type="image/png")

    audit_log.emit(
        action="media.generate",
        actor={"tenant_id": body.tenant_id, "user_email": user["email"]},
        target={"bucket": BUCKET, "blob": blob.name},
        # prompt_sha, not prompt: the audit bucket is retention-LOCKED for five years.
        meta={"model": IMAGE_MODEL, "synthid": True, "prompt_sha": key})
    log.info(json.dumps(_usage(body.tenant_id, user["email"], IMAGE_USD, False,
                               int((time.time() - t0) * 1000))))
    return {"blob": blob.name, "bucket": BUCKET, "cached": False}


@router.post("/upload-url")
def upload_url(filename: str, content_type: str, tenant_id: str,
               user=Depends(verify_iap)):
    """Past 32 MB the browser must PUT straight to GCS - into the bucket the worker watches."""
    enforce_membership(user["email"], tenant_id)
    if "/" in filename or ".." in filename or not filename:
        raise HTTPException(400, "filename must be a bare name")
    if content_type not in UPLOAD_TYPES:
        raise HTTPException(400, f"content_type must be one of {sorted(UPLOAD_TYPES)}")
    blob = _gcs.bucket(UPLOAD_BUCKET).blob(f"{tenant_id}/{filename}")
    return {"url": blob.generate_signed_url(version="v4", method="PUT",
                                            expiration=timedelta(minutes=15),
                                            content_type=content_type, **_signing_kwargs()),
            "bucket": UPLOAD_BUCKET, "blob": blob.name,
            "note": "PUT the bytes with exactly this Content-Type; the object.finalized notification "
                    "hands it to documind-ingest, and list_documents (MCP) shows it indexed."}
'''

with open('media.py', 'w') as f: f.write(MEDIA_PY)
print('media.py:', len(MEDIA_PY.splitlines()), 'lines')


In [ ]:
DOCKERFILE = '''
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends tini ca-certificates && rm -rf /var/lib/apt/lists/*
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
# BUILD CONTEXT IS deploy/, not deploy/services/rag-api/ - this service imports the answer
# contract (and, in 12.8, the IAP verifier) from shared/, so shared/ has to land in the image:
#     gcloud builds submit --config=cloudbuild.yaml .      # run from deploy/
# chat, ingest and admin already build this way.
COPY services/rag-api/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app shared/ ./shared/
COPY --chown=app:app services/rag-api/ ./
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
HEALTHCHECK --interval=20s --timeout=3s --start-period=15s CMD python -c "import urllib.request as r; r.urlopen('http://localhost:8080/health').read()" || exit 1
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["gunicorn", "-k", "uvicorn.workers.UvicornWorker", "-w", "2", "-b", "0.0.0.0:8080", "-t", "120", "--access-logfile", "-", "main:app"]
'''
with open('Dockerfile', 'w') as f: f.write(DOCKERFILE)

DEPLOY = '''
# --ingress=all, and the boundary is identity, not the network: no unauthenticated calls, a
# bearer token from the UI's service account on every request (IAM invoker), and the person's
# own IAP assertion forwarded inside it (shared/iap.py). With internal-and-cloud-load-balancing
# the UI's call - which leaves through the public run.app hostname - and the eval gate's, from
# a GitHub runner, were both refused before any token was read.
#
# The env delimiter is ^|^ because the values carry ':' (every URL) and ',' (two audiences).
# RETRIEVAL_BACKEND=vector is the deployment's default (a tenant's pin can send it to the Firestore rung
# or a managed store); the two VECTOR_ variables come from `terraform output`; SELF_URL is the deterministic run.app URL.
# DEMO_MODE=1 (Module 9): /v1/media/generate serves a cached image for a prompt it has drawn before,
# so a demo re-running one prompt is one bill; DEMO_MODE=0 make deploy-services turns it off. The two
# bucket names are storage.tf's: generated assets in media, 9.4's signed PUTs into uploads.
# AUDIT_BUCKET: the media route writes the audit row through shared/audit_log.py, which refuses to drop an
# event without a bucket - and refuses BEFORE the spend since the first live generate went 500 after it.
# Module 10: the model is a SETTING. GENERATOR_MODEL is a name (global) or a tuned endpoint path (regional,
# generator.py picks the client); RAG_MODEL_BASE prices an endpoint at its base; ROUTING=on puts router.py and
# the budget breaker on the request path; BUDGET_USD is the month's cap; SPEND_PCT overrides the reading.
# 12.5's ledger (11 September 2026): RETRIEVAL_CURRENT_ONLY=on retrieves only chunks the ledger marks current -
# after the second vector index is built and make backfill-current has run; a candidate first, like every switch.
# 12 September 2026 (deploy/INDEXING.md): EMBEDDING_MODEL / EMBEDDING_VERSION are the one declared embedding - the same
# pair the ingest worker stamps on every chunk row, so the query vector and the document vectors come from one model.
# Module 12 (12.6): THE GUARD IS A SWITCH. ARMOR=on screens the prompt before retrieval and the buffered answer after
# it against the Model Armor template (asia-south1, with the data it inspects); off on the lane, on for the candidate
# 12.6 judges - make candidate ARMOR=on.
# Module 11: THE BACKEND IS A SETTING. MODEL_BACKEND=vertex is google.genai; gateway sends the same prompt to 11.3's
# LiteLLM gateway (LITELLM_URL, the deterministic URL, behind IAM) where GENERATOR_MODEL names a route - documind-slm
# is 11.4's self-hosted model. The lane runs vertex; make candidate MODEL_BACKEND=gateway is the A/B.
# 13 September 2026: RETRIEVAL_GRAPH=off|on|auto is 4.6's graph on the lane (shared/documind_graph.py, built by make graph):
# on walks the tenant's graph for every question, auto only for a relational question with a seed entity, and the walk's
# chunks go in front of the dense pool. Off on the lane; make candidate RETRIEVAL_GRAPH=auto is where it is judged.
# GRAPH_BACKEND=firestore|spanner (16 September 2026) is WHERE that graph lives: Firestore beside the chunks, or
# spanner.tf's Spanner Graph, which seeds the walk by meaning (SPANNER_INSTANCE / SPANNER_DATABASE name it).
# P9.4 (13 September 2026): RETRIEVAL_BACKEND=rag_engine is 4.3's corpus as the retrieval stage - the mirror 12.5 keeps
# (MANAGED_MIRROR), queried by text, its contexts mapped to the kit's chunks through the ledger. The DEFAULT store: a
# tenant's own retrieval_backend pin and its data_region (tenant_settings/{tenant}, make tenant-policy) decide per request
# (the corpora are us-central1-only), so the deploy hands the API the residency and RAG_LOCATION. A candidate first:
# make candidate RETRIEVAL_BACKEND=rag_engine, after make ablate ABLATE_ARGS="--arms all" has measured it.
gcloud run deploy documind-api \\
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/api:$GIT_SHA \\
  --region=${REGION:-us-central1} --platform=managed \\
  --no-allow-unauthenticated \\
  --ingress=all \\
  --memory=1Gi --cpu=2 --concurrency=40 --timeout=120 \\
  --min-instances=0 --max-instances=20 \\
  --cpu-boost --execution-environment=gen2 \\
  --service-account=documind-api-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="^|^GOOGLE_CLOUD_PROJECT=$PROJECT|RETRIEVAL_BACKEND=${RETRIEVAL_BACKEND:-firestore}|VECTOR_INDEX_ENDPOINT=$VECTOR_INDEX_ENDPOINT|VECTOR_DEPLOYED_INDEX_ID=$VECTOR_DEPLOYED_INDEX_ID|SELF_URL=https://documind-api-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|IAP_AUDIENCE=/projects/$PROJECT_NUMBER/locations/${REGION:-us-central1}/services/documind-ui,/projects/$PROJECT_NUMBER/locations/${REGION:-us-central1}/services/documind-chat|DEMO_MODE=${DEMO_MODE-1}|UPLOAD_BUCKET=$PROJECT-uploads|MEDIA_BUCKET=$PROJECT-media|AUDIT_BUCKET=$PROJECT-audit|GENERATOR_MODEL=${GENERATOR_MODEL-gemini-3.6-flash}|RAG_MODEL_BASE=${RAG_MODEL_BASE-gemini-3.6-flash}|ROUTING=${ROUTING-off}|BUDGET_USD=${BUDGET_USD-100}${SPEND_PCT:+|SPEND_PCT=$SPEND_PCT}|MODEL_BACKEND=${MODEL_BACKEND-vertex}|LITELLM_URL=https://documind-gateway-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|ARMOR=${ARMOR-off}|ARMOR_LOCATION=${ARMOR_LOCATION-asia-south1}|ARMOR_TEMPLATE=${ARMOR_TEMPLATE-documind-guard}|SEMANTIC_CACHE=${SEMANTIC_CACHE-off}|RETRIEVAL_CURRENT_ONLY=${RETRIEVAL_CURRENT_ONLY-off}|RETRIEVAL_GRAPH=${RETRIEVAL_GRAPH-off}|GRAPH_BACKEND=${GRAPH_BACKEND-firestore}|SPANNER_INSTANCE=${SPANNER_INSTANCE-documind-graph}|SPANNER_DATABASE=${SPANNER_DATABASE-documind}|RAG_LOCATION=${RAG_LOCATION-us-central1}|EMBEDDING_MODEL=${EMBEDDING_MODEL-text-embedding-005}|EMBEDDING_VERSION=${EMBEDDING_VERSION-1}|GIT_SHA=$GIT_SHA" \\
  --vpc-connector=projects/$PROJECT/locations/${REGION:-us-central1}/connectors/documind-vpc \\
  --vpc-egress=private-ranges-only

# Who may KNOCK (12 September 2026): roles/run.invoker on THIS service, for the four identities that reach it,
# and no longer project-wide in sa.tf - project-wide admitted every one of them to every service, the A2A peer
# included. The UI's account streams answers and runs the smoke below; chat-sa and mcp-sa arrive through the
# ONE retrieve(); the outsider is the eval gate's fixture, admitted so that its refusal is the roster's 403 and
# not the network's. sa.tf's caller graph is the list, and the gate check_authz.py compares this loop with it.
for who in documind-ui-sa documind-chat-sa documind-mcp-sa documind-outsider-sa; do
  gcloud run services add-iam-policy-binding documind-api \\
    --region=${REGION:-us-central1} --project=$PROJECT \\
    --member="serviceAccount:$who@$PROJECT.iam.gserviceaccount.com" --role=roles/run.invoker --quiet
done
'''
print(DEPLOY)


In [ ]:
SMOKE = '''
# Get an identity token (not your user token) because --no-allow-unauthenticated
TOK=$(gcloud auth print-identity-token --include-email --impersonate-service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com --audiences=https://documind-api-$PROJECT_NUMBER.us-central1.run.app)
# The token is also WHO you are to rag-api (shared/iap.py's bearer leg, 12.8): documind-ui-sa must be
# on acme's roster, or the query below is a 403 - a real refusal, not a missing header.

curl -sSf -H "Authorization: Bearer $TOK" https://documind-api-xxx.run.app/health
# -> {"status":"ok"}

curl -sSf -H "Authorization: Bearer $TOK" \\
     -H "Content-Type: application/json" \\
     -d '{"query":"Which file types are supported?","tenant_id":"acme","user_id":"u_1","top_k":5,"stream":false}' \\
     https://documind-api-xxx.run.app/v1/query
# -> {"answer":"...","citations":[...],"confidence":"high","answerable":true,...}

curl -sN -H "Authorization: Bearer $TOK" \\
     -H "Content-Type: application/json" \\
     -d '{"query":"Same","tenant_id":"acme","user_id":"u_1","top_k":5}' \\
     https://documind-api-xxx.run.app/v1/stream
# -> event: citation ... event: token ... event: done
'''
print(SMOKE)
print()
INHERITS = {
    '12.3 Admin Dashboard': 'structured JSON logs (query/tenant/tokens) feed BigQuery via Log Sink',
    '12.4 Streamlit UI': 'POST /v1/stream via Server-Sent Events; citations array drives pill rendering',
    'Observability': 'Every span traced: retrieve / rerank / generate. OpenTelemetry -> Cloud Trace',
    'Tenant isolation': 'Vector Search restrict on tenant_id + server-side verify tenant matches IAP claim',
}
print('WHAT DOWNSTREAM INHERITS:')
for k, v in INHERITS.items(): print(f'  {k:28} <- {v}')
